In [1]:
import pandas_datareader.data as web #to collect data
import datetime as dt #to specify start and end dates

# import yfinance as yf

import eventstudy as es
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns


import pandas as pd

import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.regression.rolling import RollingOLS

from patsy import dmatrices
from tqdm.notebook import tqdm
tqdm.pandas()

pd.set_option('display.max_columns', 500)

In [2]:
import_folder_path = "car_output1"
output_folder_path = "car_output2"
supporting_folder_path = "supporting_datafiles"

## 120 CAR Data Reading

In [3]:
ols120 = pd.read_pickle(rf"{import_folder_path}\ols120_2.pkl")

## 120 CAR Calc

In [4]:
ols120CAR = ols120.sort_values(by = ["CompanyName", "AsOnDate"])
ols120CAR

,CompanyName,ProwessCode,Symbol,AsOnDate,ACP,pct,RF,RMRF,MF,SMB,HML,OLS120_intercept,OLS120_RMRF,OLS120_SMB,OLS120_HML,OLS120_r_squared,OLS120_adjusted_r_squared,OLS120_f_p_value
0,20 Microns Ltd.,11.0,20MICRONS,2008-10-06,16.82,NaN,0.069713,-6.381449,-6.311735,-0.373052,-0.566450,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,20 Microns Ltd.,11.0,20MICRONS,2008-10-07,15.05,-0.105232,0.023232,-0.669144,-0.645911,-1.502487,0.184699,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,20 Microns Ltd.,11.0,20MICRONS,2008-10-08,13.25,-0.119601,0.023232,-3.533362,-3.510130,-1.780674,0.072932,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,20 Microns Ltd.,11.0,20MICRONS,2008-10-10,11.60,-0.124528,0.045520,-7.052324,-7.006804,0.217126,0.354629,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,20 Microns Ltd.,11.0,20MICRONS,2008-10-13,12.32,0.062069,0.066863,5.042738,5.109602,-2.437097,0.145853,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13465088,Zylog Systems Ltd.,275793.0,ZYLOG,2024-03-21,0.35,0.000000,0.018215,1.441807,1.460021,0.572866,1.270303,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13465089,Zylog Systems Ltd.,275793.0,ZYLOG,2024-03-22,0.35,0.000000,0.018215,0.593324,0.611538,0.752002,0.124527,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13465090,Zylog Systems Ltd.,275793.0,ZYLOG,2024-03-26,0.35,0.000000,0.072878,-0.044514,0.028364,-1.091597,0.440605,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13465091,Zylog Systems Ltd.,275793.0,ZYLOG,2024-03-27,0.35,0.000000,0.018215,0.326702,0.344916,-0.165072,-0.188505,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
# FUNCTION

# def CAR120(frame):
#     if ((~frame.OLS120_intercept.isnull().any()) == True):
#         ER = frame.RF + (frame.RMRF * frame.OLS120_RMRF) + (frame.SMB * frame.OLS120_SMB) + (frame.HML * frame.OLS120_HML) + frame.OLS120_intercept
#         AR = frame.pct - ER
#         return AR.sum()
#     else: 
#         return np.NaN

    # for i in range(len(frame)):
    #     if (i>0) & ( i< len(frame)-1) :
    #         coeff120CAR = np.array([frame.iloc[i]["OLS120_RMRF"], frame.iloc[i]["OLS120_SMB"],
    #                                 frame.iloc[i]["OLS120_HML"], 1, frame.iloc[i]["OLS120_intercept"]])

    #         event120CAR3 = np.array(frame.loc[ [i-1, i, i+1], ["RMRF", "SMB", "HML", "RF"]],)
    #         event120CAR3 = [np.append(item, 1) for item in event120CAR3]

    #         ER = np.matmul(event120CAR3, coeff120CAR)

    #         actualReturns = np.array([frame.iloc[i-1]["pct"], frame.iloc[i]["pct"], frame.iloc[i+1]["pct"]])

    #         AR = np.subtract( actualReturns, ER )

    #         output120CAR3.loc[i, ["120CAR3"]] = AR.sum()


    
        # if i>=2:
        #     frame["120CAR5"].iloc[i] = CAR120(frame.iloc[ i-2 : i+2 ])
        # if i>=3:
        #     frame["120CAR7"].iloc[i] = CAR120(frame.iloc[ i-3 : i+3 ])
        # if i>=5:
        #     frame["120CAR11"].iloc[i] = CAR120(frame.iloc[ i-5 : i+5 ])

        

def CAR(frame):

    outputFrame = pd.DataFrame( index = frame.index)

        
    # 3 Days
    
    outputFrame["pct_cen3sum"] = frame["pct"].rolling(window = 3, min_periods = 3, center = True).sum()
    
    outputFrame["RF_cen3sum"] = frame["RF"].rolling(window = 3, min_periods = 3, center = True).sum()
    
    outputFrame["RMRF_cen3sum"] = frame["RMRF"].rolling(window = 3, min_periods = 3, center = True).sum()
    outputFrame["SMB_cen3sum"] = frame["SMB"].rolling(window = 3, min_periods = 3, center = True).sum()
    outputFrame["HML_cen3sum"] = frame["HML"].rolling(window = 3, min_periods = 3, center = True).sum()

    # 5 Days
    
    outputFrame["pct_cen5sum"] = frame["pct"].rolling(window = 5, min_periods = 5, center = True).sum()
    
    outputFrame["RF_cen5sum"] = frame["RF"].rolling(window = 5, min_periods = 5, center = True).sum()
    
    outputFrame["RMRF_cen5sum"] = frame["RMRF"].rolling(window = 5, min_periods = 5, center = True).sum()
    outputFrame["SMB_cen5sum"] = frame["SMB"].rolling(window = 5, min_periods = 5, center = True).sum()
    outputFrame["HML_cen5sum"] = frame["HML"].rolling(window = 5, min_periods = 5, center = True).sum()

    
    # 7 Days
    
    outputFrame["pct_cen7sum"] = frame["pct"].rolling(window = 7, min_periods = 7, center = True).sum()
    
    outputFrame["RF_cen7sum"] = frame["RF"].rolling(window = 7, min_periods = 7, center = True).sum()
    
    outputFrame["RMRF_cen7sum"] = frame["RMRF"].rolling(window = 7, min_periods = 7, center = True).sum()
    outputFrame["SMB_cen7sum"] = frame["SMB"].rolling(window = 7, min_periods = 7, center = True).sum()
    outputFrame["HML_cen7sum"] = frame["HML"].rolling(window = 7, min_periods = 7, center = True).sum()

    
    # 11 Days
    
    outputFrame["pct_cen11sum"] = frame["pct"].rolling(window = 11, min_periods = 11, center = True).sum()
    
    outputFrame["RF_cen11sum"] = frame["RF"].rolling(window = 11, min_periods = 11, center = True).sum()
    
    outputFrame["RMRF_cen11sum"] = frame["RMRF"].rolling(window = 11, min_periods = 11, center = True).sum()
    outputFrame["SMB_cen11sum"] = frame["SMB"].rolling(window = 11, min_periods = 11, center = True).sum()
    outputFrame["HML_cen11sum"] = frame["HML"].rolling(window = 11, min_periods = 11, center = True).sum()

    
    return outputFrame


In [6]:
result120CAR = ols120CAR.sort_values(by = ["CompanyName", "AsOnDate"]).groupby(by = "CompanyName").progress_apply(CAR)

  0%|          | 0/3721 [00:00<?, ?it/s]

C:\Users\SHIVAM\anaconda3\Lib\site-packages\tqdm\std.py:805: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return getattr(df, df_function)(wrapper, **kwargs)


In [7]:
result120CAR

pct_cen3sum  RF_cen3sum  RMRF_cen3sum  \
CompanyName                                                          
20 Microns Ltd.    0                 NaN         NaN           NaN   
                   1                 NaN    0.116178    -10.583954   
                   2           -0.349361    0.091985    -11.254830   
                   3           -0.182061    0.135616     -5.542948   
                   4           -0.163920    0.134667     -0.834193   
...                                  ...         ...           ...   
Zylog Systems Ltd. 13465088     0.000000    0.054643      2.222872   
                   13465089     0.000000    0.109307      1.990617   
                   13465090     0.000000    0.109307      0.875511   
                   13465091     0.000000    0.109307      0.985242   
                   13465092          NaN         NaN           NaN   

                             SMB_cen3sum  HML_cen3sum  pct_cen5sum  \
CompanyName                                                          
20 Microns Ltd.    0                 NaN          NaN          NaN   
                   1           -3.656212    -0.308820          NaN   
                   2           -3.066034     0.612259          NaN   
                   3           -4.000644     0.573414    -0.388754   
                   4           -1.442626     1.579289    -0.183251   
...                                  ...          ...          ...   
Zylog Systems Ltd. 13465088     0.647351     1.691011     0.000000   
                   13465089     0.233272     1.835435     0.000000   
                   13465090    -0.504666     0.376627     0.000000   
                   13465091    -1.836003     0.159099          NaN   
                   13465092          NaN          NaN          NaN   

                             RF_cen5sum  RMRF_cen5sum  SMB_cen5sum  \
CompanyName                                                          
20 Microns Ltd.    0                NaN           NaN          NaN   
                   1                NaN           NaN          NaN   
                   2           0.228562    -12.593540    -5.876183   
                   3           0.181132     -5.036699    -4.725787   
                   4           0.180182     -9.543586    -3.464452   
...                                 ...           ...          ...   
Zylog Systems Ltd. 13465088    0.145734      0.669402    -0.213495   
                   13465089    0.145735      2.505059    -0.609317   
                   13465090    0.145736      3.020372    -0.511135   
                   13465091         NaN           NaN          NaN   
                   13465092         NaN           NaN          NaN   

                             HML_cen5sum  pct_cen7sum  RF_cen7sum  \
CompanyName                                                         
20 Microns Ltd.    0                 NaN          NaN         NaN   
                   1                 NaN          NaN         NaN   
                   2            0.191663          NaN         NaN   
                   3            1.836919          NaN    0.273128   
                   4            1.702691    -0.163688    0.225697   
...                                  ...          ...         ...   
Zylog Systems Ltd. 13465088     1.701467     0.000000    0.218599   
                   13465089     1.943111     0.000000    0.182163   
                   13465090     1.553929          NaN         NaN   
                   13465091          NaN          NaN         NaN   
                   13465092          NaN          NaN         NaN   

                             RMRF_cen7sum  SMB_cen7sum  HML_cen7sum  \
CompanyName                                                           
20 Microns Ltd.    0                  NaN          NaN          NaN   
                   1                  NaN          NaN          NaN   
                   2                  NaN          NaN          NaN   
                   3           -16.594178  

In [8]:
result120CAR2 = result120CAR.reset_index(level = 0).drop("CompanyName", axis = 1)
result120CAR2

,pct_cen3sum,RF_cen3sum,RMRF_cen3sum,SMB_cen3sum,HML_cen3sum,pct_cen5sum,RF_cen5sum,RMRF_cen5sum,SMB_cen5sum,HML_cen5sum,pct_cen7sum,RF_cen7sum,RMRF_cen7sum,SMB_cen7sum,HML_cen7sum,pct_cen11sum,RF_cen11sum,RMRF_cen11sum,SMB_cen11sum,HML_cen11sum
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,0.116178,-10.583954,-3.656212,-0.308820,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,-0.349361,0.091985,-11.254830,-3.066034,0.612259,NaN,0.228562,-12.593540,-5.876183,0.191663,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,-0.182061,0.135616,-5.542948,-4.000644,0.573414,-0.388754,0.181132,-5.036699,-4.725787,1.836919,NaN,0.273128,-16.594178,-5.339991,1.320939,NaN,NaN,NaN,NaN,NaN
4,-0.163920,0.134667,-0.834193,-1.442626,1.579289,-0.183251,0.180182,-9.543586,-3.464452,1.702691,-0.163688,0.225697,-12.320480,-5.095004,1.701136,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13465088,0.000000,0.054643,2.222872,0.647351,1.691011,0.000000,0.145734,0.669402,-0.213495,1.701467,0.000000,0.218599,1.086124,-0.832943,1.912107,NaN,NaN,NaN,NaN,NaN
13465089,0.000000,0.109307,1.990617,0.233272,1.835435,0.000000,0.145735,2.505059,-0.609317,1.943111,0.000000,0.182163,1.699158,-0.957902,1.419960,NaN,NaN,NaN,NaN,NaN
13465090,0.000000,0.109307,0.875511,-0.504666,0.376627,0.000000,0.145736,3.020372,-0.511135,1.553929,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13465091,0.000000,0.109307,0.985242,-1.836003,0.159099,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
output120CAR = pd.concat([ols120CAR, result120CAR2], axis = 1)

In [10]:
output120CAR.columns

Index(['CompanyName', 'ProwessCode', 'Symbol', 'AsOnDate', 'ACP', 'pct', 'RF',
       'RMRF', 'MF', 'SMB', 'HML', 'OLS120_intercept', 'OLS120_RMRF',
       'OLS120_SMB', 'OLS120_HML', 'OLS120_r_squared',
       'OLS120_adjusted_r_squared', 'OLS120_f_p_value', 'pct_cen3sum',
       'RF_cen3sum', 'RMRF_cen3sum', 'SMB_cen3sum', 'HML_cen3sum',
       'pct_cen5sum', 'RF_cen5sum', 'RMRF_cen5sum', 'SMB_cen5sum',
       'HML_cen5sum', 'pct_cen7sum', 'RF_cen7sum', 'RMRF_cen7sum',
       'SMB_cen7sum', 'HML_cen7sum', 'pct_cen11sum', 'RF_cen11sum',
       'RMRF_cen11sum', 'SMB_cen11sum', 'HML_cen11sum'],
      dtype='object')

In [11]:
output120CAR["120CAR3"] = output120CAR["pct_cen3sum"] - (output120CAR["RF_cen3sum"] + 
                          
                          output120CAR["OLS120_RMRF"]*output120CAR["RMRF_cen3sum"] + 
                          output120CAR["OLS120_SMB"]*output120CAR["SMB_cen3sum"] + 
                          output120CAR["OLS120_HML"]*output120CAR["HML_cen3sum"] + 
                          output120CAR["OLS120_intercept"]*3 )


In [12]:
output120CAR["120CAR5"] = output120CAR["pct_cen5sum"] - (output120CAR["RF_cen5sum"] + 
                          
                          output120CAR["OLS120_RMRF"]*output120CAR["RMRF_cen5sum"] + 
                          output120CAR["OLS120_SMB"]*output120CAR["SMB_cen5sum"] + 
                          output120CAR["OLS120_HML"]*output120CAR["HML_cen5sum"] + 
                          output120CAR["OLS120_intercept"]*3 )

In [13]:
output120CAR["120CAR7"] = output120CAR["pct_cen7sum"] - (output120CAR["RF_cen7sum"] + 
                          
                          output120CAR["OLS120_RMRF"]*output120CAR["RMRF_cen7sum"] + 
                          output120CAR["OLS120_SMB"]*output120CAR["SMB_cen7sum"] + 
                          output120CAR["OLS120_HML"]*output120CAR["HML_cen7sum"] + 
                          output120CAR["OLS120_intercept"]*3 )

In [14]:
output120CAR["120CAR11"] = output120CAR["pct_cen11sum"] - (output120CAR["RF_cen11sum"] + 
                          
                          output120CAR["OLS120_RMRF"]*output120CAR["RMRF_cen11sum"] + 
                          output120CAR["OLS120_SMB"]*output120CAR["SMB_cen11sum"] + 
                          output120CAR["OLS120_HML"]*output120CAR["HML_cen11sum"] + 
                          output120CAR["OLS120_intercept"]*3 )

In [15]:
output120CAR

,CompanyName,ProwessCode,Symbol,AsOnDate,ACP,pct,RF,RMRF,MF,SMB,HML,OLS120_intercept,OLS120_RMRF,OLS120_SMB,OLS120_HML,OLS120_r_squared,OLS120_adjusted_r_squared,OLS120_f_p_value,pct_cen3sum,RF_cen3sum,RMRF_cen3sum,SMB_cen3sum,HML_cen3sum,pct_cen5sum,RF_cen5sum,RMRF_cen5sum,SMB_cen5sum,HML_cen5sum,pct_cen7sum,RF_cen7sum,RMRF_cen7sum,SMB_cen7sum,HML_cen7sum,pct_cen11sum,RF_cen11sum,RMRF_cen11sum,SMB_cen11sum,HML_cen11sum,120CAR3,120CAR5,120CAR7,120CAR11
0,20 Microns Ltd.,11.0,20MICRONS,2008-10-06,16.82,NaN,0.069713,-6.381449,-6.311735,-0.373052,-0.566450,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,20 Microns Ltd.,11.0,20MICRONS,2008-10-07,15.05,-0.105232,0.023232,-0.669144,-0.645911,-1.502487,0.184699,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.116178,-10.583954,-3.656212,-0.308820,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,20 Microns Ltd.,11.0,20MICRONS,2008-10-08,13.25,-0.119601,0.023232,-3.533362,-3.510130,-1.780674,0.072932,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.349361,0.091985,-11.254830,-3.066034,0.612259,NaN,0.228562,-12.593540,-5.876183,0.191663,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,20 Microns Ltd.,11.0,20MICRONS,2008-10-10,11.60,-0.124528,0.045520,-7.052324,-7.006804,0.217126,0.354629,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.182061,0.135616,-5.542948,-4.000644,0.573414,-0.388754,0.181132,-5.036699,-4.725787,1.836919,NaN,0.273128,-16.594178,-5.339991,1.320939,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,20 Microns Ltd.,11.0,20MICRONS,2008-10-13,12.32,0.062069,0.066863,5.042738,5.109602,-2.437097,0.145853,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.163920,0.134667,-0.834193,-1.442626,1.579289,-0.183251,0.180182,-9.543586,-3.464452,1.702691,-0.163688,0.225697,-12.320480,-5.095004,1.701136,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13465088,Zylog Systems Ltd.,275793.0,ZYLOG,2024-03-21,0.35,0.000000,0.018215,1.441807,1.460021,0.572866,1.270303,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.054643,2.222872,0.647351,1.691011,0.000000,0.145734,0.669402,-0.213495,1.701467,0.000000,0.218599,1.086124,-0.832943,1.912107,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13465089,Zylog Systems Ltd.,275793.0,ZYLOG,2024-03-22,0.35,0.000000,0.018215,0.593324,0.611538,0.752002,0.124527,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.109307,1.990617,0.233272,1.835435,0.000000,0.145735,2.505059,-0.609317,1.943111,0.000000,0.182163,1.699158,-0.957902,1.419960,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13465090,Zylog Systems Ltd.,275793.0,ZYLOG,2024-03-26,0.35,0.000000,0.072878,-0.044514,0.028364,-1.091597,0.440605,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.109307,0.875511,-0.504666,0.376627,0.000000,0.145736,3.020372,-0.511135,1.553929,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13465091,Zylog Systems Ltd.,275793.0,ZYLOG,2024-03-27,0.35,0.000000,0.018215,0.326702,0.344916,-0.165072,-0.188505,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.109307,0.985242,-1.836003,0.159099,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [16]:
output120CAR_2 = output120CAR.dropna(subset = "120CAR3").reset_index(drop = True).sort_values( by = ["CompanyName", "AsOnDate"] ).reset_index(drop = True)

In [17]:
output120CAR_2

,CompanyName,ProwessCode,Symbol,AsOnDate,ACP,pct,RF,RMRF,MF,SMB,HML,OLS120_intercept,OLS120_RMRF,OLS120_SMB,OLS120_HML,OLS120_r_squared,OLS120_adjusted_r_squared,OLS120_f_p_value,pct_cen3sum,RF_cen3sum,RMRF_cen3sum,SMB_cen3sum,HML_cen3sum,pct_cen5sum,RF_cen5sum,RMRF_cen5sum,SMB_cen5sum,HML_cen5sum,pct_cen7sum,RF_cen7sum,RMRF_cen7sum,SMB_cen7sum,HML_cen7sum,pct_cen11sum,RF_cen11sum,RMRF_cen11sum,SMB_cen11sum,HML_cen11sum,120CAR3,120CAR5,120CAR7,120CAR11
0,20 Microns Ltd.,11.0,20MICRONS,2009-07-29,12.60,-0.017161,0.008824,-1.187185,-1.178360,0.280266,-0.219265,-0.017647,0.005552,0.012183,0.009481,0.149680,0.127689,0.000288,-0.056087,0.026473,0.029619,1.295112,-0.491198,0.062901,0.061664,1.417488,1.518892,-0.349442,0.109739,0.096639,4.283657,1.385170,2.090903,0.181963,0.131720,6.025389,3.417154,3.499163,-0.040906,0.031116,0.005558,-0.005077
1,20 Microns Ltd.,11.0,20MICRONS,2009-07-30,12.50,-0.007937,0.008824,0.606603,0.615428,-0.050968,0.076723,-0.016498,0.005772,0.013185,0.009805,0.170255,0.148796,0.000074,0.000503,0.026365,0.353815,-0.451706,-0.282730,-0.004746,0.061340,2.473169,0.170401,0.772679,0.130467,0.096531,2.484330,1.688810,1.794841,0.143027,0.131612,4.768098,2.739052,4.060317,0.030317,-0.040692,0.029223,-0.042542
2,20 Microns Ltd.,11.0,20MICRONS,2009-08-27,15.78,0.000000,0.009040,0.358959,0.368000,0.741064,0.945923,-0.013787,0.004889,0.011979,0.005721,0.127494,0.104929,0.001196,-0.020051,0.027229,2.106517,1.916364,1.209039,0.095158,0.063718,2.095520,3.928680,2.456203,0.186179,0.099990,3.145210,4.816131,0.799632,0.367888,0.136368,5.257233,4.916455,0.078157,-0.046090,0.001445,0.049908,0.187838
3,20 Microns Ltd.,11.0,20MICRONS,2011-04-29,20.48,-0.016330,0.019851,-0.595940,-0.576089,-0.572908,-0.607305,-0.027096,0.010449,0.015732,-0.002440,0.176213,0.154908,0.000049,0.135960,0.099051,-2.055273,-1.390252,0.761902,0.068979,0.138538,-4.268887,-1.176797,-0.293157,0.048651,0.178026,-4.607642,-1.723837,0.153421,0.066696,0.316887,-5.076389,-1.348640,-0.171170,0.163401,0.074130,0.027549,-0.095063
4,20 Microns Ltd.,11.0,20MICRONS,2014-08-06,31.15,-0.011111,0.022705,-0.690542,-0.667837,1.023324,-0.182707,-0.035800,0.010683,0.006169,0.001164,0.121640,0.098924,0.001726,-0.012716,0.068115,-0.250139,1.214791,0.739524,-0.004716,0.158951,-0.558310,0.776436,0.169694,-0.002971,0.249787,-1.450041,2.056452,-0.913487,-0.040029,0.363107,-1.042710,0.259265,-3.508804,0.020884,-0.055292,-0.141493,-0.282114
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34561,Zylog Systems Ltd.,275793.0,ZYLOG,2015-08-14,5.40,0.018868,0.019213,1.546309,1.565522,-0.064032,1.416502,-0.034650,0.004784,-0.007787,0.006894,0.045213,0.020521,0.145370,-0.113635,0.096076,0.535057,-0.082231,-0.294096,-0.246220,0.134926,-0.534668,-0.050669,-1.880907,-0.291712,0.173777,-0.844522,0.077811,-2.519018,-0.274151,0.291613,-3.755096,0.048701,-3.466074,-0.106933,-0.262065,-0.339525,-0.419573
34562,Zylog Systems Ltd.,275793.0,ZYLOG,2016-06-30,3.90,0.040000,0.017832,1.065921,1.083753,-0.809196,1.214423,-0.025512,-0.001175,-0.000664,0.005110,0.016047,-0.009400,0.596696,0.080218,0.053601,2.887396,-0.810372,2.264789,0.066209,0.125044,3.865878,-0.074499,3.413269,0.082876,0.196699,3.920553,2.024007,5.684697,0.017235,0.285435,1.822980,3.816329,1.394239,0.094435,0.004753,-0.060387,-0.194113
34563,Zylog Systems Ltd.,275793.0,ZYLOG,2016-08-12,3.15,-0.045455,0.017406,0.290292,0.307698,-0.979555,0.368906,-0.028918,-0.000405,-0.002380,0.003763,0.013945,-0.011556,0.651255,-0.061816,0.104454,0.121902,-1.936704,0.287865,-0.076967,0.139266,-1.118332,-2.021379,0.410728,-0.061583,0.174078,-0.982620,-0.933242,-0.384302,-0.059889,0.313345,0.230677,-0.096989,0.458040,-0.085160,-0.136289,-0.150080,-0.288341
34564,Zylog Systems Ltd.,275793.0,ZYLOG,2016-11-23,4.25,-0.022989,0.015807,1.118596,1.134402,0.640072,1.506896,-0.028118,0.000909,-0.002034,0.008051,0.057325,0.032945,0.075918,0.0009

In [18]:
output120CAR_2.to_pickle(rf"{output_folder_path}\output120CAR_2.pkl")

In [19]:
del ols120
del ols120CAR

del result120CAR2
del result120CAR

del output120CAR
del output120CAR_2

## 150 CAR Data Reading

In [20]:
ols150 = pd.read_pickle(rf"{import_folder_path}\ols150_2.pkl")

## 150 CAR Calc

In [21]:
ols150CAR = ols150.sort_values(by = ["CompanyName", "AsOnDate"])
ols150CAR

,CompanyName,ProwessCode,Symbol,AsOnDate,ACP,pct,RF,RMRF,MF,SMB,HML,OLS150_intercept,OLS150_RMRF,OLS150_SMB,OLS150_HML,OLS150_r_squared,OLS150_adjusted_r_squared,OLS150_f_p_value
0,20 Microns Ltd.,11.0,20MICRONS,2008-10-06,16.82,NaN,0.069713,-6.381449,-6.311735,-0.373052,-0.566450,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,20 Microns Ltd.,11.0,20MICRONS,2008-10-07,15.05,-0.105232,0.023232,-0.669144,-0.645911,-1.502487,0.184699,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,20 Microns Ltd.,11.0,20MICRONS,2008-10-08,13.25,-0.119601,0.023232,-3.533362,-3.510130,-1.780674,0.072932,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,20 Microns Ltd.,11.0,20MICRONS,2008-10-10,11.60,-0.124528,0.045520,-7.052324,-7.006804,0.217126,0.354629,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,20 Microns Ltd.,11.0,20MICRONS,2008-10-13,12.32,0.062069,0.066863,5.042738,5.109602,-2.437097,0.145853,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13465088,Zylog Systems Ltd.,275793.0,ZYLOG,2024-03-21,0.35,0.000000,0.018215,1.441807,1.460021,0.572866,1.270303,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13465089,Zylog Systems Ltd.,275793.0,ZYLOG,2024-03-22,0.35,0.000000,0.018215,0.593324,0.611538,0.752002,0.124527,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13465090,Zylog Systems Ltd.,275793.0,ZYLOG,2024-03-26,0.35,0.000000,0.072878,-0.044514,0.028364,-1.091597,0.440605,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13465091,Zylog Systems Ltd.,275793.0,ZYLOG,2024-03-27,0.35,0.000000,0.018215,0.326702,0.344916,-0.165072,-0.188505,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [22]:
# FUNCTION

# def CAR150(frame):
#     if ((~frame.OLS150_intercept.isnull().any()) == True):
#         ER = frame.RF + (frame.RMRF * frame.OLS150_RMRF) + (frame.SMB * frame.OLS150_SMB) + (frame.HML * frame.OLS150_HML) + frame.OLS150_intercept
#         AR = frame.pct - ER
#         return AR.sum()
#     else: 
#         return np.NaN

    # for i in range(len(frame)):
    #     if (i>0) & ( i< len(frame)-1) :
    #         coeff150CAR = np.array([frame.iloc[i]["OLS150_RMRF"], frame.iloc[i]["OLS150_SMB"],
    #                                 frame.iloc[i]["OLS150_HML"], 1, frame.iloc[i]["OLS150_intercept"]])

    #         event150CAR3 = np.array(frame.loc[ [i-1, i, i+1], ["RMRF", "SMB", "HML", "RF"]],)
    #         event150CAR3 = [np.append(item, 1) for item in event150CAR3]

    #         ER = np.matmul(event150CAR3, coeff150CAR)

    #         actualReturns = np.array([frame.iloc[i-1]["pct"], frame.iloc[i]["pct"], frame.iloc[i+1]["pct"]])

    #         AR = np.subtract( actualReturns, ER )

    #         output150CAR3.loc[i, ["150CAR3"]] = AR.sum()


    
        # if i>=2:
        #     frame["150CAR5"].iloc[i] = CAR150(frame.iloc[ i-2 : i+2 ])
        # if i>=3:
        #     frame["150CAR7"].iloc[i] = CAR150(frame.iloc[ i-3 : i+3 ])
        # if i>=5:
        #     frame["150CAR11"].iloc[i] = CAR150(frame.iloc[ i-5 : i+5 ])

        

def CAR(frame):

    outputFrame = pd.DataFrame( index = frame.index)

        
    # 3 Days
    
    outputFrame["pct_cen3sum"] = frame["pct"].rolling(window = 3, min_periods = 3, center = True).sum()
    
    outputFrame["RF_cen3sum"] = frame["RF"].rolling(window = 3, min_periods = 3, center = True).sum()
    
    outputFrame["RMRF_cen3sum"] = frame["RMRF"].rolling(window = 3, min_periods = 3, center = True).sum()
    outputFrame["SMB_cen3sum"] = frame["SMB"].rolling(window = 3, min_periods = 3, center = True).sum()
    outputFrame["HML_cen3sum"] = frame["HML"].rolling(window = 3, min_periods = 3, center = True).sum()
    
    
    # 5 Days
    
    outputFrame["pct_cen5sum"] = frame["pct"].rolling(window = 5, min_periods = 5, center = True).sum()
    
    outputFrame["RF_cen5sum"] = frame["RF"].rolling(window = 5, min_periods = 5, center = True).sum()
    
    outputFrame["RMRF_cen5sum"] = frame["RMRF"].rolling(window = 5, min_periods = 5, center = True).sum()
    outputFrame["SMB_cen5sum"] = frame["SMB"].rolling(window = 5, min_periods = 5, center = True).sum()
    outputFrame["HML_cen5sum"] = frame["HML"].rolling(window = 5, min_periods = 5, center = True).sum()
        
    
    # 7 Days
    
    outputFrame["pct_cen7sum"] = frame["pct"].rolling(window = 7, min_periods = 7, center = True).sum()
    
    outputFrame["RF_cen7sum"] = frame["RF"].rolling(window = 7, min_periods = 7, center = True).sum()
    
    outputFrame["RMRF_cen7sum"] = frame["RMRF"].rolling(window = 7, min_periods = 7, center = True).sum()
    outputFrame["SMB_cen7sum"] = frame["SMB"].rolling(window = 7, min_periods = 7, center = True).sum()
    outputFrame["HML_cen7sum"] = frame["HML"].rolling(window = 7, min_periods = 7, center = True).sum()
        
    
    # 11 Days
    
    outputFrame["pct_cen11sum"] = frame["pct"].rolling(window = 11, min_periods = 11, center = True).sum()
    
    outputFrame["RF_cen11sum"] = frame["RF"].rolling(window = 11, min_periods = 11, center = True).sum()
    
    outputFrame["RMRF_cen11sum"] = frame["RMRF"].rolling(window = 11, min_periods = 11, center = True).sum()
    outputFrame["SMB_cen11sum"] = frame["SMB"].rolling(window = 11, min_periods = 11, center = True).sum()
    outputFrame["HML_cen11sum"] = frame["HML"].rolling(window = 11, min_periods = 11, center = True).sum()
        
    return outputFrame


In [23]:
result150CAR = ols150CAR.sort_values(by = ["CompanyName", "AsOnDate"]).groupby(by = "CompanyName").progress_apply(CAR)

  0%|          | 0/3721 [00:00<?, ?it/s]

C:\Users\SHIVAM\anaconda3\Lib\site-packages\tqdm\std.py:805: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return getattr(df, df_function)(wrapper, **kwargs)


In [24]:
result150CAR

pct_cen3sum  RF_cen3sum  RMRF_cen3sum  \
CompanyName                                                          
20 Microns Ltd.    0                 NaN         NaN           NaN   
                   1                 NaN    0.116178    -10.583954   
                   2           -0.349361    0.091985    -11.254830   
                   3           -0.182061    0.135616     -5.542948   
                   4           -0.163920    0.134667     -0.834193   
...                                  ...         ...           ...   
Zylog Systems Ltd. 13465088     0.000000    0.054643      2.222872   
                   13465089     0.000000    0.109307      1.990617   
                   13465090     0.000000    0.109307      0.875511   
                   13465091     0.000000    0.109307      0.985242   
                   13465092          NaN         NaN           NaN   

                             SMB_cen3sum  HML_cen3sum  pct_cen5sum  \
CompanyName                                                          
20 Microns Ltd.    0                 NaN          NaN          NaN   
                   1           -3.656212    -0.308820          NaN   
                   2           -3.066034     0.612259          NaN   
                   3           -4.000644     0.573414    -0.388754   
                   4           -1.442626     1.579289    -0.183251   
...                                  ...          ...          ...   
Zylog Systems Ltd. 13465088     0.647351     1.691011     0.000000   
                   13465089     0.233272     1.835435     0.000000   
                   13465090    -0.504666     0.376627     0.000000   
                   13465091    -1.836003     0.159099          NaN   
                   13465092          NaN          NaN          NaN   

                             RF_cen5sum  RMRF_cen5sum  SMB_cen5sum  \
CompanyName                                                          
20 Microns Ltd.    0                NaN           NaN          NaN   
                   1                NaN           NaN          NaN   
                   2           0.228562    -12.593540    -5.876183   
                   3           0.181132     -5.036699    -4.725787   
                   4           0.180182     -9.543586    -3.464452   
...                                 ...           ...          ...   
Zylog Systems Ltd. 13465088    0.145734      0.669402    -0.213495   
                   13465089    0.145735      2.505059    -0.609317   
                   13465090    0.145736      3.020372    -0.511135   
                   13465091         NaN           NaN          NaN   
                   13465092         NaN           NaN          NaN   

                             HML_cen5sum  pct_cen7sum  RF_cen7sum  \
CompanyName                                                         
20 Microns Ltd.    0                 NaN          NaN         NaN   
                   1                 NaN          NaN         NaN   
                   2            0.191663          NaN         NaN   
                   3            1.836919          NaN    0.273128   
                   4            1.702691    -0.163688    0.225697   
...                                  ...          ...         ...   
Zylog Systems Ltd. 13465088     1.701467     0.000000    0.218599   
                   13465089     1.943111     0.000000    0.182163   
                   13465090     1.553929          NaN         NaN   
                   13465091          NaN          NaN         NaN   
                   13465092          NaN          NaN         NaN   

                             RMRF_cen7sum  SMB_cen7sum  HML_cen7sum  \
CompanyName                                                           
20 Microns Ltd.    0                  NaN          NaN          NaN   
                   1                  NaN          NaN          NaN   
                   2                  NaN          NaN          NaN   
                   3           -16.594178  

In [25]:
result150CAR2 = result150CAR.reset_index(level = 0).drop("CompanyName", axis = 1)
result150CAR2

,pct_cen3sum,RF_cen3sum,RMRF_cen3sum,SMB_cen3sum,HML_cen3sum,pct_cen5sum,RF_cen5sum,RMRF_cen5sum,SMB_cen5sum,HML_cen5sum,pct_cen7sum,RF_cen7sum,RMRF_cen7sum,SMB_cen7sum,HML_cen7sum,pct_cen11sum,RF_cen11sum,RMRF_cen11sum,SMB_cen11sum,HML_cen11sum
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,0.116178,-10.583954,-3.656212,-0.308820,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,-0.349361,0.091985,-11.254830,-3.066034,0.612259,NaN,0.228562,-12.593540,-5.876183,0.191663,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,-0.182061,0.135616,-5.542948,-4.000644,0.573414,-0.388754,0.181132,-5.036699,-4.725787,1.836919,NaN,0.273128,-16.594178,-5.339991,1.320939,NaN,NaN,NaN,NaN,NaN
4,-0.163920,0.134667,-0.834193,-1.442626,1.579289,-0.183251,0.180182,-9.543586,-3.464452,1.702691,-0.163688,0.225697,-12.320480,-5.095004,1.701136,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13465088,0.000000,0.054643,2.222872,0.647351,1.691011,0.000000,0.145734,0.669402,-0.213495,1.701467,0.000000,0.218599,1.086124,-0.832943,1.912107,NaN,NaN,NaN,NaN,NaN
13465089,0.000000,0.109307,1.990617,0.233272,1.835435,0.000000,0.145735,2.505059,-0.609317,1.943111,0.000000,0.182163,1.699158,-0.957902,1.419960,NaN,NaN,NaN,NaN,NaN
13465090,0.000000,0.109307,0.875511,-0.504666,0.376627,0.000000,0.145736,3.020372,-0.511135,1.553929,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13465091,0.000000,0.109307,0.985242,-1.836003,0.159099,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [26]:
output150CAR = pd.concat([ols150CAR, result150CAR2], axis = 1)
output150CAR

,CompanyName,ProwessCode,Symbol,AsOnDate,ACP,pct,RF,RMRF,MF,SMB,HML,OLS150_intercept,OLS150_RMRF,OLS150_SMB,OLS150_HML,OLS150_r_squared,OLS150_adjusted_r_squared,OLS150_f_p_value,pct_cen3sum,RF_cen3sum,RMRF_cen3sum,SMB_cen3sum,HML_cen3sum,pct_cen5sum,RF_cen5sum,RMRF_cen5sum,SMB_cen5sum,HML_cen5sum,pct_cen7sum,RF_cen7sum,RMRF_cen7sum,SMB_cen7sum,HML_cen7sum,pct_cen11sum,RF_cen11sum,RMRF_cen11sum,SMB_cen11sum,HML_cen11sum
0,20 Microns Ltd.,11.0,20MICRONS,2008-10-06,16.82,NaN,0.069713,-6.381449,-6.311735,-0.373052,-0.566450,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,20 Microns Ltd.,11.0,20MICRONS,2008-10-07,15.05,-0.105232,0.023232,-0.669144,-0.645911,-1.502487,0.184699,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.116178,-10.583954,-3.656212,-0.308820,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,20 Microns Ltd.,11.0,20MICRONS,2008-10-08,13.25,-0.119601,0.023232,-3.533362,-3.510130,-1.780674,0.072932,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.349361,0.091985,-11.254830,-3.066034,0.612259,NaN,0.228562,-12.593540,-5.876183,0.191663,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,20 Microns Ltd.,11.0,20MICRONS,2008-10-10,11.60,-0.124528,0.045520,-7.052324,-7.006804,0.217126,0.354629,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.182061,0.135616,-5.542948,-4.000644,0.573414,-0.388754,0.181132,-5.036699,-4.725787,1.836919,NaN,0.273128,-16.594178,-5.339991,1.320939,NaN,NaN,NaN,NaN,NaN
4,20 Microns Ltd.,11.0,20MICRONS,2008-10-13,12.32,0.062069,0.066863,5.042738,5.109602,-2.437097,0.145853,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.163920,0.134667,-0.834193,-1.442626,1.579289,-0.183251,0.180182,-9.543586,-3.464452,1.702691,-0.163688,0.225697,-12.320480,-5.095004,1.701136,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13465088,Zylog Systems Ltd.,275793.0,ZYLOG,2024-03-21,0.35,0.000000,0.018215,1.441807,1.460021,0.572866,1.270303,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.054643,2.222872,0.647351,1.691011,0.000000,0.145734,0.669402,-0.213495,1.701467,0.000000,0.218599,1.086124,-0.832943,1.912107,NaN,NaN,NaN,NaN,NaN
13465089,Zylog Systems Ltd.,275793.0,ZYLOG,2024-03-22,0.35,0.000000,0.018215,0.593324,0.611538,0.752002,0.124527,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.109307,1.990617,0.233272,1.835435,0.000000,0.145735,2.505059,-0.609317,1.943111,0.000000,0.182163,1.699158,-0.957902,1.419960,NaN,NaN,NaN,NaN,NaN
13465090,Zylog Systems Ltd.,275793.0,ZYLOG,2024-03-26,0.35,0.000000,0.072878,-0.044514,0.028364,-1.091597,0.440605,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.109307,0.875511,-0.504666,0.376627,0.000000,0.145736,3.020372,-0.511135,1.553929,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13465091,Zylog Systems Ltd.,275793.0,ZYLOG,2024-03-27,0.35,0.000000,0.018215,0.326702,0.344916,-0.165072,-0.188505,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.109307,0.985242,-1.836003,0.159099,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [27]:
output150CAR.columns

Index(['CompanyName', 'ProwessCode', 'Symbol', 'AsOnDate', 'ACP', 'pct', 'RF',
       'RMRF', 'MF', 'SMB', 'HML', 'OLS150_intercept', 'OLS150_RMRF',
       'OLS150_SMB', 'OLS150_HML', 'OLS150_r_squared',
       'OLS150_adjusted_r_squared', 'OLS150_f_p_value', 'pct_cen3sum',
       'RF_cen3sum', 'RMRF_cen3sum', 'SMB_cen3sum', 'HML_cen3sum',
       'pct_cen5sum', 'RF_cen5sum', 'RMRF_cen5sum', 'SMB_cen5sum',
       'HML_cen5sum', 'pct_cen7sum', 'RF_cen7sum', 'RMRF_cen7sum',
       'SMB_cen7sum', 'HML_cen7sum', 'pct_cen11sum', 'RF_cen11sum',
       'RMRF_cen11sum', 'SMB_cen11sum', 'HML_cen11sum'],
      dtype='object')

In [28]:
output150CAR["150CAR3"] = output150CAR["pct_cen3sum"] - (output150CAR["RF_cen3sum"] + 
                          
                          output150CAR["OLS150_RMRF"]*output150CAR["RMRF_cen3sum"] + 
                          output150CAR["OLS150_SMB"]*output150CAR["SMB_cen3sum"] + 
                          output150CAR["OLS150_HML"]*output150CAR["HML_cen3sum"] + 
                          output150CAR["OLS150_intercept"]*3 )

In [29]:
output150CAR["150CAR5"] = output150CAR["pct_cen5sum"] - (output150CAR["RF_cen5sum"] + 
                          
                          output150CAR["OLS150_RMRF"]*output150CAR["RMRF_cen5sum"] + 
                          output150CAR["OLS150_SMB"]*output150CAR["SMB_cen5sum"] + 
                          output150CAR["OLS150_HML"]*output150CAR["HML_cen5sum"] + 
                          output150CAR["OLS150_intercept"]*3 )

In [30]:
output150CAR["150CAR7"] = output150CAR["pct_cen7sum"] - (output150CAR["RF_cen7sum"] + 
                          
                          output150CAR["OLS150_RMRF"]*output150CAR["RMRF_cen7sum"] + 
                          output150CAR["OLS150_SMB"]*output150CAR["SMB_cen7sum"] + 
                          output150CAR["OLS150_HML"]*output150CAR["HML_cen7sum"] + 
                          output150CAR["OLS150_intercept"]*3 )

In [31]:
output150CAR["150CAR11"] = output150CAR["pct_cen11sum"] - (output150CAR["RF_cen11sum"] + 
                          
                          output150CAR["OLS150_RMRF"]*output150CAR["RMRF_cen11sum"] + 
                          output150CAR["OLS150_SMB"]*output150CAR["SMB_cen11sum"] + 
                          output150CAR["OLS150_HML"]*output150CAR["HML_cen11sum"] + 
                          output150CAR["OLS150_intercept"]*3 )

In [32]:
output150CAR

,CompanyName,ProwessCode,Symbol,AsOnDate,ACP,pct,RF,RMRF,MF,SMB,HML,OLS150_intercept,OLS150_RMRF,OLS150_SMB,OLS150_HML,OLS150_r_squared,OLS150_adjusted_r_squared,OLS150_f_p_value,pct_cen3sum,RF_cen3sum,RMRF_cen3sum,SMB_cen3sum,HML_cen3sum,pct_cen5sum,RF_cen5sum,RMRF_cen5sum,SMB_cen5sum,HML_cen5sum,pct_cen7sum,RF_cen7sum,RMRF_cen7sum,SMB_cen7sum,HML_cen7sum,pct_cen11sum,RF_cen11sum,RMRF_cen11sum,SMB_cen11sum,HML_cen11sum,150CAR3,150CAR5,150CAR7,150CAR11
0,20 Microns Ltd.,11.0,20MICRONS,2008-10-06,16.82,NaN,0.069713,-6.381449,-6.311735,-0.373052,-0.566450,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,20 Microns Ltd.,11.0,20MICRONS,2008-10-07,15.05,-0.105232,0.023232,-0.669144,-0.645911,-1.502487,0.184699,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.116178,-10.583954,-3.656212,-0.308820,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,20 Microns Ltd.,11.0,20MICRONS,2008-10-08,13.25,-0.119601,0.023232,-3.533362,-3.510130,-1.780674,0.072932,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.349361,0.091985,-11.254830,-3.066034,0.612259,NaN,0.228562,-12.593540,-5.876183,0.191663,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,20 Microns Ltd.,11.0,20MICRONS,2008-10-10,11.60,-0.124528,0.045520,-7.052324,-7.006804,0.217126,0.354629,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.182061,0.135616,-5.542948,-4.000644,0.573414,-0.388754,0.181132,-5.036699,-4.725787,1.836919,NaN,0.273128,-16.594178,-5.339991,1.320939,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,20 Microns Ltd.,11.0,20MICRONS,2008-10-13,12.32,0.062069,0.066863,5.042738,5.109602,-2.437097,0.145853,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.163920,0.134667,-0.834193,-1.442626,1.579289,-0.183251,0.180182,-9.543586,-3.464452,1.702691,-0.163688,0.225697,-12.320480,-5.095004,1.701136,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13465088,Zylog Systems Ltd.,275793.0,ZYLOG,2024-03-21,0.35,0.000000,0.018215,1.441807,1.460021,0.572866,1.270303,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.054643,2.222872,0.647351,1.691011,0.000000,0.145734,0.669402,-0.213495,1.701467,0.000000,0.218599,1.086124,-0.832943,1.912107,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13465089,Zylog Systems Ltd.,275793.0,ZYLOG,2024-03-22,0.35,0.000000,0.018215,0.593324,0.611538,0.752002,0.124527,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.109307,1.990617,0.233272,1.835435,0.000000,0.145735,2.505059,-0.609317,1.943111,0.000000,0.182163,1.699158,-0.957902,1.419960,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13465090,Zylog Systems Ltd.,275793.0,ZYLOG,2024-03-26,0.35,0.000000,0.072878,-0.044514,0.028364,-1.091597,0.440605,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.109307,0.875511,-0.504666,0.376627,0.000000,0.145736,3.020372,-0.511135,1.553929,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13465091,Zylog Systems Ltd.,275793.0,ZYLOG,2024-03-27,0.35,0.000000,0.018215,0.326702,0.344916,-0.165072,-0.188505,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.109307,0.985242,-1.836003,0.159099,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [33]:
output150CAR_2 = output150CAR.dropna(subset = "150CAR3").reset_index(drop = True)
output150CAR_2

,CompanyName,ProwessCode,Symbol,AsOnDate,ACP,pct,RF,RMRF,MF,SMB,HML,OLS150_intercept,OLS150_RMRF,OLS150_SMB,OLS150_HML,OLS150_r_squared,OLS150_adjusted_r_squared,OLS150_f_p_value,pct_cen3sum,RF_cen3sum,RMRF_cen3sum,SMB_cen3sum,HML_cen3sum,pct_cen5sum,RF_cen5sum,RMRF_cen5sum,SMB_cen5sum,HML_cen5sum,pct_cen7sum,RF_cen7sum,RMRF_cen7sum,SMB_cen7sum,HML_cen7sum,pct_cen11sum,RF_cen11sum,RMRF_cen11sum,SMB_cen11sum,HML_cen11sum,150CAR3,150CAR5,150CAR7,150CAR11
0,20 Microns Ltd.,11.0,20MICRONS,2009-07-29,12.60,-0.017161,0.008824,-1.187185,-1.178360,0.280266,-0.219265,-0.018884,0.007822,0.012183,0.011318,0.181844,0.165033,0.000002,-0.056087,0.026473,0.029619,1.295112,-0.491198,0.062901,0.061664,1.417488,1.518892,-0.349442,0.109739,0.096639,4.283657,1.385170,2.090903,0.181963,0.131720,6.025389,3.417154,3.499163,-0.036358,0.032253,-0.004294,-0.021468
1,20 Microns Ltd.,11.0,20MICRONS,2009-07-30,12.50,-0.007937,0.008824,0.606603,0.615428,-0.050968,0.076723,-0.017772,0.007442,0.011800,0.008235,0.158646,0.141357,0.000013,0.000503,0.026365,0.353815,-0.451706,-0.282730,-0.004746,0.061340,2.473169,0.170401,0.772679,0.130467,0.096531,2.484330,1.688810,1.794841,0.143027,0.131612,4.768098,2.739052,4.060317,0.032478,-0.039550,0.034055,-0.036511
2,20 Microns Ltd.,11.0,20MICRONS,2009-08-27,15.78,0.000000,0.009040,0.358959,0.368000,0.741064,0.945923,-0.017060,0.006434,0.011588,0.008780,0.163067,0.145870,0.000009,-0.020051,0.027229,2.106517,1.916364,1.209039,0.095158,0.063718,2.095520,3.928680,2.456203,0.186179,0.099990,3.145210,4.816131,0.799632,0.367888,0.136368,5.257233,4.916455,0.078157,-0.042477,0.002045,0.054302,0.191216
3,20 Microns Ltd.,11.0,20MICRONS,2011-04-29,20.48,-0.016330,0.019851,-0.595940,-0.576089,-0.572908,-0.607305,-0.025992,0.008764,0.012305,0.002310,0.158002,0.140700,0.000014,0.135960,0.099051,-2.055273,-1.390252,0.761902,0.068979,0.138538,-4.268887,-1.176797,-0.293157,0.048651,0.178026,-4.607642,-1.723837,0.153421,0.066696,0.316887,-5.076389,-1.348640,-0.171170,0.148246,0.060988,0.009841,-0.110733
4,20 Microns Ltd.,11.0,20MICRONS,2014-08-06,31.15,-0.011111,0.022705,-0.690542,-0.667837,1.023324,-0.182707,-0.035619,0.008433,0.006457,0.001327,0.084912,0.066109,0.004638,-0.012716,0.068115,-0.250139,1.214791,0.739524,-0.004716,0.158951,-0.558310,0.776436,0.169694,-0.002971,0.249787,-1.450041,2.056452,-0.913487,-0.040029,0.363107,-1.042710,0.259265,-3.508804,0.019308,-0.057342,-0.145741,-0.284504
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34393,Zylog Systems Ltd.,275793.0,ZYLOG,2015-08-14,5.40,0.018868,0.019213,1.546309,1.565522,-0.064032,1.416502,-0.034041,0.006057,-0.003198,0.008756,0.079905,0.060999,0.006729,-0.113635,0.096076,0.535057,-0.082231,-0.294096,-0.246220,0.134926,-0.534668,-0.050669,-1.880907,-0.291712,0.173777,-0.844522,0.077811,-2.519018,-0.274151,0.291613,-3.755096,0.048701,-3.466074,-0.108517,-0.259477,-0.335944,-0.410390
34394,Zylog Systems Ltd.,275793.0,ZYLOG,2016-06-30,3.90,0.040000,0.017832,1.065921,1.083753,-0.809196,1.214423,-0.027929,0.000399,0.000096,0.003071,0.007924,-0.012461,0.761294,0.080218,0.053601,2.887396,-0.810372,2.264789,0.066209,0.125044,3.865878,-0.074499,3.413269,0.082876,0.196699,3.920553,2.024007,5.684697,0.017235,0.285435,1.822980,3.816329,1.394239,0.102376,0.012937,-0.049250,-0.189788
34395,Zylog Systems Ltd.,275793.0,ZYLOG,2016-08-12,3.15,-0.045455,0.017406,0.290292,0.307698,-0.979555,0.368906,-0.025870,-0.001230,-0.001257,0.006396,0.025822,0.005804,0.280130,-0.061816,0.104454,0.121902,-1.936704,0.287865,-0.076967,0.139266,-1.118332,-2.021379,0.410728,-0.061583,0.174078,-0.982620,-0.933242,-0.384302,-0.059889,0.313345,0.230677,-0.096989,0.458040,-0.092785,-0.145167,-0.157975,-0.298391
34396,Zylog Systems Ltd.,275793.0,ZYLOG,2016-11-23,4.25,-0.022989,0.015807,1.118596,1.134402,0.640072,1.506896,-0.029026,0.001455,-0.002229,0.005384,0.035742,0.015929,0.149041,0.000961,

In [34]:
output150CAR_2.to_pickle(rf"{output_folder_path}\output150CAR_2.pkl")

In [35]:
del ols150
del ols150CAR

del result150CAR2
del result150CAR

del output150CAR
del output150CAR_2

## 180 CAR Data Reading

In [36]:
ols180 = pd.read_pickle(rf"{import_folder_path}\ols180_2.pkl")

## 180 CAR Calc

In [37]:
ols180CAR = ols180.sort_values(by = ["CompanyName", "AsOnDate"])
ols180CAR

,CompanyName,ProwessCode,Symbol,AsOnDate,ACP,pct,RF,RMRF,MF,SMB,HML,OLS180_intercept,OLS180_RMRF,OLS180_SMB,OLS180_HML,OLS180_r_squared,OLS180_adjusted_r_squared,OLS180_f_p_value
0,20 Microns Ltd.,11.0,20MICRONS,2008-10-06,16.82,NaN,0.069713,-6.381449,-6.311735,-0.373052,-0.566450,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,20 Microns Ltd.,11.0,20MICRONS,2008-10-07,15.05,-0.105232,0.023232,-0.669144,-0.645911,-1.502487,0.184699,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,20 Microns Ltd.,11.0,20MICRONS,2008-10-08,13.25,-0.119601,0.023232,-3.533362,-3.510130,-1.780674,0.072932,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,20 Microns Ltd.,11.0,20MICRONS,2008-10-10,11.60,-0.124528,0.045520,-7.052324,-7.006804,0.217126,0.354629,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,20 Microns Ltd.,11.0,20MICRONS,2008-10-13,12.32,0.062069,0.066863,5.042738,5.109602,-2.437097,0.145853,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13465088,Zylog Systems Ltd.,275793.0,ZYLOG,2024-03-21,0.35,0.000000,0.018215,1.441807,1.460021,0.572866,1.270303,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13465089,Zylog Systems Ltd.,275793.0,ZYLOG,2024-03-22,0.35,0.000000,0.018215,0.593324,0.611538,0.752002,0.124527,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13465090,Zylog Systems Ltd.,275793.0,ZYLOG,2024-03-26,0.35,0.000000,0.072878,-0.044514,0.028364,-1.091597,0.440605,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13465091,Zylog Systems Ltd.,275793.0,ZYLOG,2024-03-27,0.35,0.000000,0.018215,0.326702,0.344916,-0.165072,-0.188505,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [38]:
# FUNCTION

# def CAR180(frame):
#     if ((~frame.OLS180_intercept.isnull().any()) == True):
#         ER = frame.RF + (frame.RMRF * frame.OLS180_RMRF) + (frame.SMB * frame.OLS180_SMB) + (frame.HML * frame.OLS180_HML) + frame.OLS180_intercept
#         AR = frame.pct - ER
#         return AR.sum()
#     else: 
#         return np.NaN

    # for i in range(len(frame)):
    #     if (i>0) & ( i< len(frame)-1) :
    #         coeff180CAR = np.array([frame.iloc[i]["OLS180_RMRF"], frame.iloc[i]["OLS180_SMB"],
    #                                 frame.iloc[i]["OLS180_HML"], 1, frame.iloc[i]["OLS180_intercept"]])

    #         event180CAR3 = np.array(frame.loc[ [i-1, i, i+1], ["RMRF", "SMB", "HML", "RF"]],)
    #         event180CAR3 = [np.append(item, 1) for item in event180CAR3]

    #         ER = np.matmul(event180CAR3, coeff180CAR)

    #         actualReturns = np.array([frame.iloc[i-1]["pct"], frame.iloc[i]["pct"], frame.iloc[i+1]["pct"]])

    #         AR = np.subtract( actualReturns, ER )

    #         output180CAR3.loc[i, ["180CAR3"]] = AR.sum()


    
        # if i>=2:
        #     frame["180CAR5"].iloc[i] = CAR180(frame.iloc[ i-2 : i+2 ])
        # if i>=3:
        #     frame["180CAR7"].iloc[i] = CAR180(frame.iloc[ i-3 : i+3 ])
        # if i>=5:
        #     frame["180CAR11"].iloc[i] = CAR180(frame.iloc[ i-5 : i+5 ])

        

def CAR(frame):

    outputFrame = pd.DataFrame( index = frame.index)

        
    # 3 Days
    
    outputFrame["pct_cen3sum"] = frame["pct"].rolling(window = 3, min_periods = 3, center = True).sum()
    
    outputFrame["RF_cen3sum"] = frame["RF"].rolling(window = 3, min_periods = 3, center = True).sum()
    
    outputFrame["RMRF_cen3sum"] = frame["RMRF"].rolling(window = 3, min_periods = 3, center = True).sum()
    outputFrame["SMB_cen3sum"] = frame["SMB"].rolling(window = 3, min_periods = 3, center = True).sum()
    outputFrame["HML_cen3sum"] = frame["HML"].rolling(window = 3, min_periods = 3, center = True).sum()
    
    
    # 5 Days
    
    outputFrame["pct_cen5sum"] = frame["pct"].rolling(window = 5, min_periods = 5, center = True).sum()
    
    outputFrame["RF_cen5sum"] = frame["RF"].rolling(window = 5, min_periods = 5, center = True).sum()
    
    outputFrame["RMRF_cen5sum"] = frame["RMRF"].rolling(window = 5, min_periods = 5, center = True).sum()
    outputFrame["SMB_cen5sum"] = frame["SMB"].rolling(window = 5, min_periods = 5, center = True).sum()
    outputFrame["HML_cen5sum"] = frame["HML"].rolling(window = 5, min_periods = 5, center = True).sum()
    
    
    # 7 Days
    
    outputFrame["pct_cen7sum"] = frame["pct"].rolling(window = 7, min_periods = 7, center = True).sum()
    
    outputFrame["RF_cen7sum"] = frame["RF"].rolling(window = 7, min_periods = 7, center = True).sum()
    
    outputFrame["RMRF_cen7sum"] = frame["RMRF"].rolling(window = 7, min_periods = 7, center = True).sum()
    outputFrame["SMB_cen7sum"] = frame["SMB"].rolling(window = 7, min_periods = 7, center = True).sum()
    outputFrame["HML_cen7sum"] = frame["HML"].rolling(window = 7, min_periods = 7, center = True).sum()
    
    
    # 11 Days
    
    outputFrame["pct_cen11sum"] = frame["pct"].rolling(window = 11, min_periods = 11, center = True).sum()
    
    outputFrame["RF_cen11sum"] = frame["RF"].rolling(window = 11, min_periods = 11, center = True).sum()
    
    outputFrame["RMRF_cen11sum"] = frame["RMRF"].rolling(window = 11, min_periods = 11, center = True).sum()
    outputFrame["SMB_cen11sum"] = frame["SMB"].rolling(window = 11, min_periods = 11, center = True).sum()
    outputFrame["HML_cen11sum"] = frame["HML"].rolling(window = 11, min_periods = 11, center = True).sum()
    
    
    return outputFrame


In [39]:
result180CAR = ols180CAR.sort_values(by = ["CompanyName", "AsOnDate"]).groupby(by = "CompanyName").progress_apply(CAR)

  0%|          | 0/3721 [00:00<?, ?it/s]

C:\Users\SHIVAM\anaconda3\Lib\site-packages\tqdm\std.py:805: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return getattr(df, df_function)(wrapper, **kwargs)


In [40]:
result180CAR

pct_cen3sum  RF_cen3sum  RMRF_cen3sum  \
CompanyName                                                          
20 Microns Ltd.    0                 NaN         NaN           NaN   
                   1                 NaN    0.116178    -10.583954   
                   2           -0.349361    0.091985    -11.254830   
                   3           -0.182061    0.135616     -5.542948   
                   4           -0.163920    0.134667     -0.834193   
...                                  ...         ...           ...   
Zylog Systems Ltd. 13465088     0.000000    0.054643      2.222872   
                   13465089     0.000000    0.109307      1.990617   
                   13465090     0.000000    0.109307      0.875511   
                   13465091     0.000000    0.109307      0.985242   
                   13465092          NaN         NaN           NaN   

                             SMB_cen3sum  HML_cen3sum  pct_cen5sum  \
CompanyName                                                          
20 Microns Ltd.    0                 NaN          NaN          NaN   
                   1           -3.656212    -0.308820          NaN   
                   2           -3.066034     0.612259          NaN   
                   3           -4.000644     0.573414    -0.388754   
                   4           -1.442626     1.579289    -0.183251   
...                                  ...          ...          ...   
Zylog Systems Ltd. 13465088     0.647351     1.691011     0.000000   
                   13465089     0.233272     1.835435     0.000000   
                   13465090    -0.504666     0.376627     0.000000   
                   13465091    -1.836003     0.159099          NaN   
                   13465092          NaN          NaN          NaN   

                             RF_cen5sum  RMRF_cen5sum  SMB_cen5sum  \
CompanyName                                                          
20 Microns Ltd.    0                NaN           NaN          NaN   
                   1                NaN           NaN          NaN   
                   2           0.228562    -12.593540    -5.876183   
                   3           0.181132     -5.036699    -4.725787   
                   4           0.180182     -9.543586    -3.464452   
...                                 ...           ...          ...   
Zylog Systems Ltd. 13465088    0.145734      0.669402    -0.213495   
                   13465089    0.145735      2.505059    -0.609317   
                   13465090    0.145736      3.020372    -0.511135   
                   13465091         NaN           NaN          NaN   
                   13465092         NaN           NaN          NaN   

                             HML_cen5sum  pct_cen7sum  RF_cen7sum  \
CompanyName                                                         
20 Microns Ltd.    0                 NaN          NaN         NaN   
                   1                 NaN          NaN         NaN   
                   2            0.191663          NaN         NaN   
                   3            1.836919          NaN    0.273128   
                   4            1.702691    -0.163688    0.225697   
...                                  ...          ...         ...   
Zylog Systems Ltd. 13465088     1.701467     0.000000    0.218599   
                   13465089     1.943111     0.000000    0.182163   
                   13465090     1.553929          NaN         NaN   
                   13465091          NaN          NaN         NaN   
                   13465092          NaN          NaN         NaN   

                             RMRF_cen7sum  SMB_cen7sum  HML_cen7sum  \
CompanyName                                                           
20 Microns Ltd.    0                  NaN          NaN          NaN   
                   1                  NaN          NaN          NaN   
                   2                  NaN          NaN          NaN   
                   3           -16.594178  

In [41]:
result180CAR2 = result180CAR.reset_index(level = 0).drop("CompanyName", axis = 1)
result180CAR2

,pct_cen3sum,RF_cen3sum,RMRF_cen3sum,SMB_cen3sum,HML_cen3sum,pct_cen5sum,RF_cen5sum,RMRF_cen5sum,SMB_cen5sum,HML_cen5sum,pct_cen7sum,RF_cen7sum,RMRF_cen7sum,SMB_cen7sum,HML_cen7sum,pct_cen11sum,RF_cen11sum,RMRF_cen11sum,SMB_cen11sum,HML_cen11sum
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,0.116178,-10.583954,-3.656212,-0.308820,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,-0.349361,0.091985,-11.254830,-3.066034,0.612259,NaN,0.228562,-12.593540,-5.876183,0.191663,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,-0.182061,0.135616,-5.542948,-4.000644,0.573414,-0.388754,0.181132,-5.036699,-4.725787,1.836919,NaN,0.273128,-16.594178,-5.339991,1.320939,NaN,NaN,NaN,NaN,NaN
4,-0.163920,0.134667,-0.834193,-1.442626,1.579289,-0.183251,0.180182,-9.543586,-3.464452,1.702691,-0.163688,0.225697,-12.320480,-5.095004,1.701136,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13465088,0.000000,0.054643,2.222872,0.647351,1.691011,0.000000,0.145734,0.669402,-0.213495,1.701467,0.000000,0.218599,1.086124,-0.832943,1.912107,NaN,NaN,NaN,NaN,NaN
13465089,0.000000,0.109307,1.990617,0.233272,1.835435,0.000000,0.145735,2.505059,-0.609317,1.943111,0.000000,0.182163,1.699158,-0.957902,1.419960,NaN,NaN,NaN,NaN,NaN
13465090,0.000000,0.109307,0.875511,-0.504666,0.376627,0.000000,0.145736,3.020372,-0.511135,1.553929,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13465091,0.000000,0.109307,0.985242,-1.836003,0.159099,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [42]:
output180CAR = pd.concat([ols180CAR, result180CAR2], axis = 1)
output180CAR

,CompanyName,ProwessCode,Symbol,AsOnDate,ACP,pct,RF,RMRF,MF,SMB,HML,OLS180_intercept,OLS180_RMRF,OLS180_SMB,OLS180_HML,OLS180_r_squared,OLS180_adjusted_r_squared,OLS180_f_p_value,pct_cen3sum,RF_cen3sum,RMRF_cen3sum,SMB_cen3sum,HML_cen3sum,pct_cen5sum,RF_cen5sum,RMRF_cen5sum,SMB_cen5sum,HML_cen5sum,pct_cen7sum,RF_cen7sum,RMRF_cen7sum,SMB_cen7sum,HML_cen7sum,pct_cen11sum,RF_cen11sum,RMRF_cen11sum,SMB_cen11sum,HML_cen11sum
0,20 Microns Ltd.,11.0,20MICRONS,2008-10-06,16.82,NaN,0.069713,-6.381449,-6.311735,-0.373052,-0.566450,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,20 Microns Ltd.,11.0,20MICRONS,2008-10-07,15.05,-0.105232,0.023232,-0.669144,-0.645911,-1.502487,0.184699,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.116178,-10.583954,-3.656212,-0.308820,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,20 Microns Ltd.,11.0,20MICRONS,2008-10-08,13.25,-0.119601,0.023232,-3.533362,-3.510130,-1.780674,0.072932,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.349361,0.091985,-11.254830,-3.066034,0.612259,NaN,0.228562,-12.593540,-5.876183,0.191663,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,20 Microns Ltd.,11.0,20MICRONS,2008-10-10,11.60,-0.124528,0.045520,-7.052324,-7.006804,0.217126,0.354629,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.182061,0.135616,-5.542948,-4.000644,0.573414,-0.388754,0.181132,-5.036699,-4.725787,1.836919,NaN,0.273128,-16.594178,-5.339991,1.320939,NaN,NaN,NaN,NaN,NaN
4,20 Microns Ltd.,11.0,20MICRONS,2008-10-13,12.32,0.062069,0.066863,5.042738,5.109602,-2.437097,0.145853,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.163920,0.134667,-0.834193,-1.442626,1.579289,-0.183251,0.180182,-9.543586,-3.464452,1.702691,-0.163688,0.225697,-12.320480,-5.095004,1.701136,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13465088,Zylog Systems Ltd.,275793.0,ZYLOG,2024-03-21,0.35,0.000000,0.018215,1.441807,1.460021,0.572866,1.270303,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.054643,2.222872,0.647351,1.691011,0.000000,0.145734,0.669402,-0.213495,1.701467,0.000000,0.218599,1.086124,-0.832943,1.912107,NaN,NaN,NaN,NaN,NaN
13465089,Zylog Systems Ltd.,275793.0,ZYLOG,2024-03-22,0.35,0.000000,0.018215,0.593324,0.611538,0.752002,0.124527,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.109307,1.990617,0.233272,1.835435,0.000000,0.145735,2.505059,-0.609317,1.943111,0.000000,0.182163,1.699158,-0.957902,1.419960,NaN,NaN,NaN,NaN,NaN
13465090,Zylog Systems Ltd.,275793.0,ZYLOG,2024-03-26,0.35,0.000000,0.072878,-0.044514,0.028364,-1.091597,0.440605,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.109307,0.875511,-0.504666,0.376627,0.000000,0.145736,3.020372,-0.511135,1.553929,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13465091,Zylog Systems Ltd.,275793.0,ZYLOG,2024-03-27,0.35,0.000000,0.018215,0.326702,0.344916,-0.165072,-0.188505,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.109307,0.985242,-1.836003,0.159099,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [43]:
output180CAR.columns

Index(['CompanyName', 'ProwessCode', 'Symbol', 'AsOnDate', 'ACP', 'pct', 'RF',
       'RMRF', 'MF', 'SMB', 'HML', 'OLS180_intercept', 'OLS180_RMRF',
       'OLS180_SMB', 'OLS180_HML', 'OLS180_r_squared',
       'OLS180_adjusted_r_squared', 'OLS180_f_p_value', 'pct_cen3sum',
       'RF_cen3sum', 'RMRF_cen3sum', 'SMB_cen3sum', 'HML_cen3sum',
       'pct_cen5sum', 'RF_cen5sum', 'RMRF_cen5sum', 'SMB_cen5sum',
       'HML_cen5sum', 'pct_cen7sum', 'RF_cen7sum', 'RMRF_cen7sum',
       'SMB_cen7sum', 'HML_cen7sum', 'pct_cen11sum', 'RF_cen11sum',
       'RMRF_cen11sum', 'SMB_cen11sum', 'HML_cen11sum'],
      dtype='object')

In [44]:
output180CAR["180CAR3"] = output180CAR["pct_cen3sum"] - (output180CAR["RF_cen3sum"] + 
                          
                          output180CAR["OLS180_RMRF"]*output180CAR["RMRF_cen3sum"] + 
                          output180CAR["OLS180_SMB"]*output180CAR["SMB_cen3sum"] + 
                          output180CAR["OLS180_HML"]*output180CAR["HML_cen3sum"] + 
                          output180CAR["OLS180_intercept"]*3 )


In [45]:
output180CAR["180CAR5"] = output180CAR["pct_cen5sum"] - (output180CAR["RF_cen5sum"] + 
                          
                          output180CAR["OLS180_RMRF"]*output180CAR["RMRF_cen5sum"] + 
                          output180CAR["OLS180_SMB"]*output180CAR["SMB_cen5sum"] + 
                          output180CAR["OLS180_HML"]*output180CAR["HML_cen5sum"] + 
                          output180CAR["OLS180_intercept"]*3 )

In [46]:
output180CAR["180CAR7"] = output180CAR["pct_cen7sum"] - (output180CAR["RF_cen7sum"] + 
                          
                          output180CAR["OLS180_RMRF"]*output180CAR["RMRF_cen7sum"] + 
                          output180CAR["OLS180_SMB"]*output180CAR["SMB_cen7sum"] + 
                          output180CAR["OLS180_HML"]*output180CAR["HML_cen7sum"] + 
                          output180CAR["OLS180_intercept"]*3 )

In [47]:
output180CAR["180CAR11"] = output180CAR["pct_cen11sum"] - (output180CAR["RF_cen11sum"] + 
                          
                          output180CAR["OLS180_RMRF"]*output180CAR["RMRF_cen11sum"] + 
                          output180CAR["OLS180_SMB"]*output180CAR["SMB_cen11sum"] + 
                          output180CAR["OLS180_HML"]*output180CAR["HML_cen11sum"] + 
                          output180CAR["OLS180_intercept"]*3 )

In [48]:
output180CAR

,CompanyName,ProwessCode,Symbol,AsOnDate,ACP,pct,RF,RMRF,MF,SMB,HML,OLS180_intercept,OLS180_RMRF,OLS180_SMB,OLS180_HML,OLS180_r_squared,OLS180_adjusted_r_squared,OLS180_f_p_value,pct_cen3sum,RF_cen3sum,RMRF_cen3sum,SMB_cen3sum,HML_cen3sum,pct_cen5sum,RF_cen5sum,RMRF_cen5sum,SMB_cen5sum,HML_cen5sum,pct_cen7sum,RF_cen7sum,RMRF_cen7sum,SMB_cen7sum,HML_cen7sum,pct_cen11sum,RF_cen11sum,RMRF_cen11sum,SMB_cen11sum,HML_cen11sum,180CAR3,180CAR5,180CAR7,180CAR11
0,20 Microns Ltd.,11.0,20MICRONS,2008-10-06,16.82,NaN,0.069713,-6.381449,-6.311735,-0.373052,-0.566450,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,20 Microns Ltd.,11.0,20MICRONS,2008-10-07,15.05,-0.105232,0.023232,-0.669144,-0.645911,-1.502487,0.184699,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.116178,-10.583954,-3.656212,-0.308820,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,20 Microns Ltd.,11.0,20MICRONS,2008-10-08,13.25,-0.119601,0.023232,-3.533362,-3.510130,-1.780674,0.072932,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.349361,0.091985,-11.254830,-3.066034,0.612259,NaN,0.228562,-12.593540,-5.876183,0.191663,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,20 Microns Ltd.,11.0,20MICRONS,2008-10-10,11.60,-0.124528,0.045520,-7.052324,-7.006804,0.217126,0.354629,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.182061,0.135616,-5.542948,-4.000644,0.573414,-0.388754,0.181132,-5.036699,-4.725787,1.836919,NaN,0.273128,-16.594178,-5.339991,1.320939,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,20 Microns Ltd.,11.0,20MICRONS,2008-10-13,12.32,0.062069,0.066863,5.042738,5.109602,-2.437097,0.145853,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.163920,0.134667,-0.834193,-1.442626,1.579289,-0.183251,0.180182,-9.543586,-3.464452,1.702691,-0.163688,0.225697,-12.320480,-5.095004,1.701136,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13465088,Zylog Systems Ltd.,275793.0,ZYLOG,2024-03-21,0.35,0.000000,0.018215,1.441807,1.460021,0.572866,1.270303,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.054643,2.222872,0.647351,1.691011,0.000000,0.145734,0.669402,-0.213495,1.701467,0.000000,0.218599,1.086124,-0.832943,1.912107,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13465089,Zylog Systems Ltd.,275793.0,ZYLOG,2024-03-22,0.35,0.000000,0.018215,0.593324,0.611538,0.752002,0.124527,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.109307,1.990617,0.233272,1.835435,0.000000,0.145735,2.505059,-0.609317,1.943111,0.000000,0.182163,1.699158,-0.957902,1.419960,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13465090,Zylog Systems Ltd.,275793.0,ZYLOG,2024-03-26,0.35,0.000000,0.072878,-0.044514,0.028364,-1.091597,0.440605,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.109307,0.875511,-0.504666,0.376627,0.000000,0.145736,3.020372,-0.511135,1.553929,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13465091,Zylog Systems Ltd.,275793.0,ZYLOG,2024-03-27,0.35,0.000000,0.018215,0.326702,0.344916,-0.165072,-0.188505,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.109307,0.985242,-1.836003,0.159099,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [49]:
output180CAR_2 = output180CAR.dropna(subset = "180CAR3").reset_index(drop = True)
output180CAR_2

,CompanyName,ProwessCode,Symbol,AsOnDate,ACP,pct,RF,RMRF,MF,SMB,HML,OLS180_intercept,OLS180_RMRF,OLS180_SMB,OLS180_HML,OLS180_r_squared,OLS180_adjusted_r_squared,OLS180_f_p_value,pct_cen3sum,RF_cen3sum,RMRF_cen3sum,SMB_cen3sum,HML_cen3sum,pct_cen5sum,RF_cen5sum,RMRF_cen5sum,SMB_cen5sum,HML_cen5sum,pct_cen7sum,RF_cen7sum,RMRF_cen7sum,SMB_cen7sum,HML_cen7sum,pct_cen11sum,RF_cen11sum,RMRF_cen11sum,SMB_cen11sum,HML_cen11sum,180CAR3,180CAR5,180CAR7,180CAR11
0,20 Microns Ltd.,11.0,20MICRONS,2009-08-27,15.78,0.000000,0.009040,0.358959,0.368000,0.741064,0.945923,-0.019523,0.008528,0.010563,0.007012,0.162659,0.148386,7.238692e-07,-0.020051,0.027229,2.106517,1.916364,1.209039,0.095158,0.063718,2.095520,3.928680,2.456203,0.186179,0.099990,3.145210,4.816131,0.799632,0.367888,0.136368,5.257233,4.916455,0.078157,-0.035395,0.013419,0.061457,0.192774
1,20 Microns Ltd.,11.0,20MICRONS,2011-04-29,20.48,-0.016330,0.019851,-0.595940,-0.576089,-0.572908,-0.607305,-0.023910,0.008059,0.012570,0.004786,0.128802,0.113952,2.125338e-05,0.135960,0.099051,-2.055273,-1.390252,0.761902,0.068979,0.138538,-4.268887,-1.176797,-0.293157,0.048651,0.178026,-4.607642,-1.723837,0.153421,0.066696,0.316887,-5.076389,-1.348640,-0.171170,0.139034,0.052771,0.000425,-0.119776
2,20 Microns Ltd.,11.0,20MICRONS,2014-08-06,31.15,-0.011111,0.022705,-0.690542,-0.667837,1.023324,-0.182707,-0.035592,0.008052,0.006294,0.002610,0.076637,0.060898,2.808170e-03,-0.012716,0.068115,-0.250139,1.214791,0.739524,-0.004716,0.158951,-0.558310,0.776436,0.169694,-0.002971,0.249787,-1.450041,2.056452,-0.913487,-0.040029,0.363107,-1.042710,0.259265,-3.508804,0.018383,-0.057725,-0.144865,-0.280436
3,20 Microns Ltd.,11.0,20MICRONS,2015-02-05,38.50,-0.020356,0.021755,-0.625772,-0.604017,0.012769,-1.213916,-0.035151,0.007464,0.003937,-0.002033,0.042711,0.026393,5.254045e-02,0.001722,0.065159,-1.736416,-0.804964,0.077852,-0.022663,0.152086,-2.821734,-1.126323,-2.075115,-0.020077,0.238803,-2.753260,0.046695,-2.249996,-0.086138,0.325611,-0.871457,0.052846,1.148929,0.058303,-0.048021,-0.137637,-0.297664
4,20 Microns Ltd.,11.0,20MICRONS,2017-05-04,43.70,0.016279,0.016554,0.046274,0.062828,0.831535,-0.581499,-0.023958,0.016109,0.010135,0.001335,0.219310,0.206003,1.752317e-09,-0.023826,0.049554,-0.994942,1.668469,-2.628765,-0.031524,0.165027,-0.704433,2.261335,-2.673886,-0.043378,0.198027,-0.312208,1.999131,-0.730319,-0.078551,0.263922,0.266193,0.943764,-0.758803,0.001120,-0.132678,-0.183787,-0.283439
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34082,Zylog Systems Ltd.,275793.0,ZYLOG,2015-08-14,5.40,0.018868,0.019213,1.546309,1.565522,-0.064032,1.416502,-0.033714,0.005668,-0.003566,0.012798,0.106314,0.091081,1.833697e-04,-0.113635,0.096076,0.535057,-0.082231,-0.294096,-0.246220,0.134926,-0.534668,-0.050669,-1.880907,-0.291712,0.173777,-0.844522,0.077811,-2.519018,-0.274151,0.291613,-3.755096,0.048701,-3.466074,-0.108131,-0.253083,-0.327043,-0.398804
34083,Zylog Systems Ltd.,275793.0,ZYLOG,2016-06-30,3.90,0.040000,0.017832,1.065921,1.083753,-0.809196,1.214423,-0.029250,0.003714,-0.001238,0.002913,0.026072,0.009471,1.982265e-01,0.080218,0.053601,2.887396,-0.810372,2.264789,0.066209,0.125044,3.865878,-0.074499,3.413269,0.082876,0.196699,3.920553,2.024007,5.684697,0.017235,0.285435,1.822980,3.816329,1.394239,0.096045,0.004526,-0.054684,-0.186557
34084,Zylog Systems Ltd.,275793.0,ZYLOG,2016-08-12,3.15,-0.045455,0.017406,0.290292,0.307698,-0.979555,0.368906,-0.027848,0.000326,-0.000422,0.004325,0.014731,-0.002063,4.541101e-01,-0.061816,0.104454,0.121902,-1.936704,0.287865,-0.076967,0.139266,-1.118332,-2.021379,0.410728,-0.061583,0.174078,-0.982620,-0.933242,-0.384302,-0.059889,0.313345,0.230677,-0.096989,0.458040,-0.084829,-0.134955,-0.150529,-0.291787
34085,Zylog Systems Ltd.,275793.0,ZYLOG,2016-11-23,4.25,-0.022989,0.015807,1.118596,1.134402,0.640072,1.506896,-0.030886,0.002208,-0.003

In [50]:
output180CAR_2.to_pickle(rf"{output_folder_path}\output180CAR_2.pkl")

In [51]:
del ols180
del ols180CAR

del result180CAR2
del result180CAR

del output180CAR
del output180CAR_2

## 210 CAR Data Reading

In [52]:
ols210 = pd.read_pickle(rf"{import_folder_path}\ols210_2.pkl")

## 210 CAR Calc

In [53]:
ols210CAR = ols210.sort_values(by = ["CompanyName", "AsOnDate"])
ols210CAR

,CompanyName,ProwessCode,Symbol,AsOnDate,ACP,pct,RF,RMRF,MF,SMB,HML,OLS210_intercept,OLS210_RMRF,OLS210_SMB,OLS210_HML,OLS210_r_squared,OLS210_adjusted_r_squared,OLS210_f_p_value
0,20 Microns Ltd.,11.0,20MICRONS,2008-10-06,16.82,NaN,0.069713,-6.381449,-6.311735,-0.373052,-0.566450,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,20 Microns Ltd.,11.0,20MICRONS,2008-10-07,15.05,-0.105232,0.023232,-0.669144,-0.645911,-1.502487,0.184699,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,20 Microns Ltd.,11.0,20MICRONS,2008-10-08,13.25,-0.119601,0.023232,-3.533362,-3.510130,-1.780674,0.072932,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,20 Microns Ltd.,11.0,20MICRONS,2008-10-10,11.60,-0.124528,0.045520,-7.052324,-7.006804,0.217126,0.354629,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,20 Microns Ltd.,11.0,20MICRONS,2008-10-13,12.32,0.062069,0.066863,5.042738,5.109602,-2.437097,0.145853,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13465088,Zylog Systems Ltd.,275793.0,ZYLOG,2024-03-21,0.35,0.000000,0.018215,1.441807,1.460021,0.572866,1.270303,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13465089,Zylog Systems Ltd.,275793.0,ZYLOG,2024-03-22,0.35,0.000000,0.018215,0.593324,0.611538,0.752002,0.124527,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13465090,Zylog Systems Ltd.,275793.0,ZYLOG,2024-03-26,0.35,0.000000,0.072878,-0.044514,0.028364,-1.091597,0.440605,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13465091,Zylog Systems Ltd.,275793.0,ZYLOG,2024-03-27,0.35,0.000000,0.018215,0.326702,0.344916,-0.165072,-0.188505,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [54]:
# FUNCTION

# def CAR210(frame):
#     if ((~frame.OLS210_intercept.isnull().any()) == True):
#         ER = frame.RF + (frame.RMRF * frame.OLS210_RMRF) + (frame.SMB * frame.OLS210_SMB) + (frame.HML * frame.OLS210_HML) + frame.OLS210_intercept
#         AR = frame.pct - ER
#         return AR.sum()
#     else: 
#         return np.NaN

    # for i in range(len(frame)):
    #     if (i>0) & ( i< len(frame)-1) :
    #         coeff210CAR = np.array([frame.iloc[i]["OLS210_RMRF"], frame.iloc[i]["OLS210_SMB"],
    #                                 frame.iloc[i]["OLS210_HML"], 1, frame.iloc[i]["OLS210_intercept"]])

    #         event210CAR3 = np.array(frame.loc[ [i-1, i, i+1], ["RMRF", "SMB", "HML", "RF"]],)
    #         event210CAR3 = [np.append(item, 1) for item in event210CAR3]

    #         ER = np.matmul(event210CAR3, coeff210CAR)

    #         actualReturns = np.array([frame.iloc[i-1]["pct"], frame.iloc[i]["pct"], frame.iloc[i+1]["pct"]])

    #         AR = np.subtract( actualReturns, ER )

    #         output210CAR3.loc[i, ["210CAR3"]] = AR.sum()


    
        # if i>=2:
        #     frame["210CAR5"].iloc[i] = CAR210(frame.iloc[ i-2 : i+2 ])
        # if i>=3:
        #     frame["210CAR7"].iloc[i] = CAR210(frame.iloc[ i-3 : i+3 ])
        # if i>=5:
        #     frame["210CAR11"].iloc[i] = CAR210(frame.iloc[ i-5 : i+5 ])

        

def CAR(frame):

    outputFrame = pd.DataFrame( index = frame.index)

        
    # 3 Days
    
    outputFrame["pct_cen3sum"] = frame["pct"].rolling(window = 3, min_periods = 3, center = True).sum()
    
    outputFrame["RF_cen3sum"] = frame["RF"].rolling(window = 3, min_periods = 3, center = True).sum()
    
    outputFrame["RMRF_cen3sum"] = frame["RMRF"].rolling(window = 3, min_periods = 3, center = True).sum()
    outputFrame["SMB_cen3sum"] = frame["SMB"].rolling(window = 3, min_periods = 3, center = True).sum()
    outputFrame["HML_cen3sum"] = frame["HML"].rolling(window = 3, min_periods = 3, center = True).sum()
    
    
    # 5 Days
    
    outputFrame["pct_cen5sum"] = frame["pct"].rolling(window = 5, min_periods = 5, center = True).sum()
    
    outputFrame["RF_cen5sum"] = frame["RF"].rolling(window = 5, min_periods = 5, center = True).sum()
    
    outputFrame["RMRF_cen5sum"] = frame["RMRF"].rolling(window = 5, min_periods = 5, center = True).sum()
    outputFrame["SMB_cen5sum"] = frame["SMB"].rolling(window = 5, min_periods = 5, center = True).sum()
    outputFrame["HML_cen5sum"] = frame["HML"].rolling(window = 5, min_periods = 5, center = True).sum()
    
    
    # 7 Days
    
    outputFrame["pct_cen7sum"] = frame["pct"].rolling(window = 7, min_periods = 7, center = True).sum()
    
    outputFrame["RF_cen7sum"] = frame["RF"].rolling(window = 7, min_periods = 7, center = True).sum()
    
    outputFrame["RMRF_cen7sum"] = frame["RMRF"].rolling(window = 7, min_periods = 7, center = True).sum()
    outputFrame["SMB_cen7sum"] = frame["SMB"].rolling(window = 7, min_periods = 7, center = True).sum()
    outputFrame["HML_cen7sum"] = frame["HML"].rolling(window = 7, min_periods = 7, center = True).sum()
    
    
    # 11 Days
    
    outputFrame["pct_cen11sum"] = frame["pct"].rolling(window = 11, min_periods = 11, center = True).sum()
    
    outputFrame["RF_cen11sum"] = frame["RF"].rolling(window = 11, min_periods = 11, center = True).sum()
    
    outputFrame["RMRF_cen11sum"] = frame["RMRF"].rolling(window = 11, min_periods = 11, center = True).sum()
    outputFrame["SMB_cen11sum"] = frame["SMB"].rolling(window = 11, min_periods = 11, center = True).sum()
    outputFrame["HML_cen11sum"] = frame["HML"].rolling(window = 11, min_periods = 11, center = True).sum()
    
    
    return outputFrame


In [55]:
result210CAR = ols210CAR.sort_values(by = ["CompanyName", "AsOnDate"]).groupby(by = "CompanyName").progress_apply(CAR)

  0%|          | 0/3721 [00:00<?, ?it/s]

C:\Users\SHIVAM\anaconda3\Lib\site-packages\tqdm\std.py:805: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return getattr(df, df_function)(wrapper, **kwargs)


In [56]:
result210CAR

pct_cen3sum  RF_cen3sum  RMRF_cen3sum  \
CompanyName                                                          
20 Microns Ltd.    0                 NaN         NaN           NaN   
                   1                 NaN    0.116178    -10.583954   
                   2           -0.349361    0.091985    -11.254830   
                   3           -0.182061    0.135616     -5.542948   
                   4           -0.163920    0.134667     -0.834193   
...                                  ...         ...           ...   
Zylog Systems Ltd. 13465088     0.000000    0.054643      2.222872   
                   13465089     0.000000    0.109307      1.990617   
                   13465090     0.000000    0.109307      0.875511   
                   13465091     0.000000    0.109307      0.985242   
                   13465092          NaN         NaN           NaN   

                             SMB_cen3sum  HML_cen3sum  pct_cen5sum  \
CompanyName                                                          
20 Microns Ltd.    0                 NaN          NaN          NaN   
                   1           -3.656212    -0.308820          NaN   
                   2           -3.066034     0.612259          NaN   
                   3           -4.000644     0.573414    -0.388754   
                   4           -1.442626     1.579289    -0.183251   
...                                  ...          ...          ...   
Zylog Systems Ltd. 13465088     0.647351     1.691011     0.000000   
                   13465089     0.233272     1.835435     0.000000   
                   13465090    -0.504666     0.376627     0.000000   
                   13465091    -1.836003     0.159099          NaN   
                   13465092          NaN          NaN          NaN   

                             RF_cen5sum  RMRF_cen5sum  SMB_cen5sum  \
CompanyName                                                          
20 Microns Ltd.    0                NaN           NaN          NaN   
                   1                NaN           NaN          NaN   
                   2           0.228562    -12.593540    -5.876183   
                   3           0.181132     -5.036699    -4.725787   
                   4           0.180182     -9.543586    -3.464452   
...                                 ...           ...          ...   
Zylog Systems Ltd. 13465088    0.145734      0.669402    -0.213495   
                   13465089    0.145735      2.505059    -0.609317   
                   13465090    0.145736      3.020372    -0.511135   
                   13465091         NaN           NaN          NaN   
                   13465092         NaN           NaN          NaN   

                             HML_cen5sum  pct_cen7sum  RF_cen7sum  \
CompanyName                                                         
20 Microns Ltd.    0                 NaN          NaN         NaN   
                   1                 NaN          NaN         NaN   
                   2            0.191663          NaN         NaN   
                   3            1.836919          NaN    0.273128   
                   4            1.702691    -0.163688    0.225697   
...                                  ...          ...         ...   
Zylog Systems Ltd. 13465088     1.701467     0.000000    0.218599   
                   13465089     1.943111     0.000000    0.182163   
                   13465090     1.553929          NaN         NaN   
                   13465091          NaN          NaN         NaN   
                   13465092          NaN          NaN         NaN   

                             RMRF_cen7sum  SMB_cen7sum  HML_cen7sum  \
CompanyName                                                           
20 Microns Ltd.    0                  NaN          NaN          NaN   
                   1                  NaN          NaN          NaN   
                   2                  NaN          NaN          NaN   
                   3           -16.594178  

In [57]:
result210CAR2 = result210CAR.reset_index(level = 0).drop("CompanyName", axis = 1)
result210CAR2

,pct_cen3sum,RF_cen3sum,RMRF_cen3sum,SMB_cen3sum,HML_cen3sum,pct_cen5sum,RF_cen5sum,RMRF_cen5sum,SMB_cen5sum,HML_cen5sum,pct_cen7sum,RF_cen7sum,RMRF_cen7sum,SMB_cen7sum,HML_cen7sum,pct_cen11sum,RF_cen11sum,RMRF_cen11sum,SMB_cen11sum,HML_cen11sum
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,0.116178,-10.583954,-3.656212,-0.308820,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,-0.349361,0.091985,-11.254830,-3.066034,0.612259,NaN,0.228562,-12.593540,-5.876183,0.191663,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,-0.182061,0.135616,-5.542948,-4.000644,0.573414,-0.388754,0.181132,-5.036699,-4.725787,1.836919,NaN,0.273128,-16.594178,-5.339991,1.320939,NaN,NaN,NaN,NaN,NaN
4,-0.163920,0.134667,-0.834193,-1.442626,1.579289,-0.183251,0.180182,-9.543586,-3.464452,1.702691,-0.163688,0.225697,-12.320480,-5.095004,1.701136,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13465088,0.000000,0.054643,2.222872,0.647351,1.691011,0.000000,0.145734,0.669402,-0.213495,1.701467,0.000000,0.218599,1.086124,-0.832943,1.912107,NaN,NaN,NaN,NaN,NaN
13465089,0.000000,0.109307,1.990617,0.233272,1.835435,0.000000,0.145735,2.505059,-0.609317,1.943111,0.000000,0.182163,1.699158,-0.957902,1.419960,NaN,NaN,NaN,NaN,NaN
13465090,0.000000,0.109307,0.875511,-0.504666,0.376627,0.000000,0.145736,3.020372,-0.511135,1.553929,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13465091,0.000000,0.109307,0.985242,-1.836003,0.159099,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [58]:
output210CAR = pd.concat([ols210CAR, result210CAR2], axis = 1)
output210CAR

,CompanyName,ProwessCode,Symbol,AsOnDate,ACP,pct,RF,RMRF,MF,SMB,HML,OLS210_intercept,OLS210_RMRF,OLS210_SMB,OLS210_HML,OLS210_r_squared,OLS210_adjusted_r_squared,OLS210_f_p_value,pct_cen3sum,RF_cen3sum,RMRF_cen3sum,SMB_cen3sum,HML_cen3sum,pct_cen5sum,RF_cen5sum,RMRF_cen5sum,SMB_cen5sum,HML_cen5sum,pct_cen7sum,RF_cen7sum,RMRF_cen7sum,SMB_cen7sum,HML_cen7sum,pct_cen11sum,RF_cen11sum,RMRF_cen11sum,SMB_cen11sum,HML_cen11sum
0,20 Microns Ltd.,11.0,20MICRONS,2008-10-06,16.82,NaN,0.069713,-6.381449,-6.311735,-0.373052,-0.566450,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,20 Microns Ltd.,11.0,20MICRONS,2008-10-07,15.05,-0.105232,0.023232,-0.669144,-0.645911,-1.502487,0.184699,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.116178,-10.583954,-3.656212,-0.308820,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,20 Microns Ltd.,11.0,20MICRONS,2008-10-08,13.25,-0.119601,0.023232,-3.533362,-3.510130,-1.780674,0.072932,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.349361,0.091985,-11.254830,-3.066034,0.612259,NaN,0.228562,-12.593540,-5.876183,0.191663,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,20 Microns Ltd.,11.0,20MICRONS,2008-10-10,11.60,-0.124528,0.045520,-7.052324,-7.006804,0.217126,0.354629,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.182061,0.135616,-5.542948,-4.000644,0.573414,-0.388754,0.181132,-5.036699,-4.725787,1.836919,NaN,0.273128,-16.594178,-5.339991,1.320939,NaN,NaN,NaN,NaN,NaN
4,20 Microns Ltd.,11.0,20MICRONS,2008-10-13,12.32,0.062069,0.066863,5.042738,5.109602,-2.437097,0.145853,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.163920,0.134667,-0.834193,-1.442626,1.579289,-0.183251,0.180182,-9.543586,-3.464452,1.702691,-0.163688,0.225697,-12.320480,-5.095004,1.701136,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13465088,Zylog Systems Ltd.,275793.0,ZYLOG,2024-03-21,0.35,0.000000,0.018215,1.441807,1.460021,0.572866,1.270303,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.054643,2.222872,0.647351,1.691011,0.000000,0.145734,0.669402,-0.213495,1.701467,0.000000,0.218599,1.086124,-0.832943,1.912107,NaN,NaN,NaN,NaN,NaN
13465089,Zylog Systems Ltd.,275793.0,ZYLOG,2024-03-22,0.35,0.000000,0.018215,0.593324,0.611538,0.752002,0.124527,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.109307,1.990617,0.233272,1.835435,0.000000,0.145735,2.505059,-0.609317,1.943111,0.000000,0.182163,1.699158,-0.957902,1.419960,NaN,NaN,NaN,NaN,NaN
13465090,Zylog Systems Ltd.,275793.0,ZYLOG,2024-03-26,0.35,0.000000,0.072878,-0.044514,0.028364,-1.091597,0.440605,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.109307,0.875511,-0.504666,0.376627,0.000000,0.145736,3.020372,-0.511135,1.553929,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13465091,Zylog Systems Ltd.,275793.0,ZYLOG,2024-03-27,0.35,0.000000,0.018215,0.326702,0.344916,-0.165072,-0.188505,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.109307,0.985242,-1.836003,0.159099,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [59]:
output210CAR.columns

Index(['CompanyName', 'ProwessCode', 'Symbol', 'AsOnDate', 'ACP', 'pct', 'RF',
       'RMRF', 'MF', 'SMB', 'HML', 'OLS210_intercept', 'OLS210_RMRF',
       'OLS210_SMB', 'OLS210_HML', 'OLS210_r_squared',
       'OLS210_adjusted_r_squared', 'OLS210_f_p_value', 'pct_cen3sum',
       'RF_cen3sum', 'RMRF_cen3sum', 'SMB_cen3sum', 'HML_cen3sum',
       'pct_cen5sum', 'RF_cen5sum', 'RMRF_cen5sum', 'SMB_cen5sum',
       'HML_cen5sum', 'pct_cen7sum', 'RF_cen7sum', 'RMRF_cen7sum',
       'SMB_cen7sum', 'HML_cen7sum', 'pct_cen11sum', 'RF_cen11sum',
       'RMRF_cen11sum', 'SMB_cen11sum', 'HML_cen11sum'],
      dtype='object')

In [60]:
output210CAR["210CAR3"] = output210CAR["pct_cen3sum"] - (output210CAR["RF_cen3sum"] + 
                          
                          output210CAR["OLS210_RMRF"]*output210CAR["RMRF_cen3sum"] + 
                          output210CAR["OLS210_SMB"]*output210CAR["SMB_cen3sum"] + 
                          output210CAR["OLS210_HML"]*output210CAR["HML_cen3sum"] + 
                          output210CAR["OLS210_intercept"]*3 )


In [61]:
output210CAR["210CAR5"] = output210CAR["pct_cen5sum"] - (output210CAR["RF_cen5sum"] + 
                          
                          output210CAR["OLS210_RMRF"]*output210CAR["RMRF_cen5sum"] + 
                          output210CAR["OLS210_SMB"]*output210CAR["SMB_cen5sum"] + 
                          output210CAR["OLS210_HML"]*output210CAR["HML_cen5sum"] + 
                          output210CAR["OLS210_intercept"]*3 )

In [62]:
output210CAR["210CAR7"] = output210CAR["pct_cen7sum"] - (output210CAR["RF_cen7sum"] + 
                          
                          output210CAR["OLS210_RMRF"]*output210CAR["RMRF_cen7sum"] + 
                          output210CAR["OLS210_SMB"]*output210CAR["SMB_cen7sum"] + 
                          output210CAR["OLS210_HML"]*output210CAR["HML_cen7sum"] + 
                          output210CAR["OLS210_intercept"]*3 )

In [63]:
output210CAR["210CAR11"] = output210CAR["pct_cen11sum"] - (output210CAR["RF_cen11sum"] + 
                          
                          output210CAR["OLS210_RMRF"]*output210CAR["RMRF_cen11sum"] + 
                          output210CAR["OLS210_SMB"]*output210CAR["SMB_cen11sum"] + 
                          output210CAR["OLS210_HML"]*output210CAR["HML_cen11sum"] + 
                          output210CAR["OLS210_intercept"]*3 )

In [64]:
output210CAR

,CompanyName,ProwessCode,Symbol,AsOnDate,ACP,pct,RF,RMRF,MF,SMB,HML,OLS210_intercept,OLS210_RMRF,OLS210_SMB,OLS210_HML,OLS210_r_squared,OLS210_adjusted_r_squared,OLS210_f_p_value,pct_cen3sum,RF_cen3sum,RMRF_cen3sum,SMB_cen3sum,HML_cen3sum,pct_cen5sum,RF_cen5sum,RMRF_cen5sum,SMB_cen5sum,HML_cen5sum,pct_cen7sum,RF_cen7sum,RMRF_cen7sum,SMB_cen7sum,HML_cen7sum,pct_cen11sum,RF_cen11sum,RMRF_cen11sum,SMB_cen11sum,HML_cen11sum,210CAR3,210CAR5,210CAR7,210CAR11
0,20 Microns Ltd.,11.0,20MICRONS,2008-10-06,16.82,NaN,0.069713,-6.381449,-6.311735,-0.373052,-0.566450,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,20 Microns Ltd.,11.0,20MICRONS,2008-10-07,15.05,-0.105232,0.023232,-0.669144,-0.645911,-1.502487,0.184699,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.116178,-10.583954,-3.656212,-0.308820,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,20 Microns Ltd.,11.0,20MICRONS,2008-10-08,13.25,-0.119601,0.023232,-3.533362,-3.510130,-1.780674,0.072932,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.349361,0.091985,-11.254830,-3.066034,0.612259,NaN,0.228562,-12.593540,-5.876183,0.191663,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,20 Microns Ltd.,11.0,20MICRONS,2008-10-10,11.60,-0.124528,0.045520,-7.052324,-7.006804,0.217126,0.354629,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.182061,0.135616,-5.542948,-4.000644,0.573414,-0.388754,0.181132,-5.036699,-4.725787,1.836919,NaN,0.273128,-16.594178,-5.339991,1.320939,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,20 Microns Ltd.,11.0,20MICRONS,2008-10-13,12.32,0.062069,0.066863,5.042738,5.109602,-2.437097,0.145853,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.163920,0.134667,-0.834193,-1.442626,1.579289,-0.183251,0.180182,-9.543586,-3.464452,1.702691,-0.163688,0.225697,-12.320480,-5.095004,1.701136,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13465088,Zylog Systems Ltd.,275793.0,ZYLOG,2024-03-21,0.35,0.000000,0.018215,1.441807,1.460021,0.572866,1.270303,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.054643,2.222872,0.647351,1.691011,0.000000,0.145734,0.669402,-0.213495,1.701467,0.000000,0.218599,1.086124,-0.832943,1.912107,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13465089,Zylog Systems Ltd.,275793.0,ZYLOG,2024-03-22,0.35,0.000000,0.018215,0.593324,0.611538,0.752002,0.124527,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.109307,1.990617,0.233272,1.835435,0.000000,0.145735,2.505059,-0.609317,1.943111,0.000000,0.182163,1.699158,-0.957902,1.419960,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13465090,Zylog Systems Ltd.,275793.0,ZYLOG,2024-03-26,0.35,0.000000,0.072878,-0.044514,0.028364,-1.091597,0.440605,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.109307,0.875511,-0.504666,0.376627,0.000000,0.145736,3.020372,-0.511135,1.553929,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13465091,Zylog Systems Ltd.,275793.0,ZYLOG,2024-03-27,0.35,0.000000,0.018215,0.326702,0.344916,-0.165072,-0.188505,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.109307,0.985242,-1.836003,0.159099,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [65]:
output210CAR_2 = output210CAR.dropna(subset = "210CAR3").reset_index(drop = True)
output210CAR_2

,CompanyName,ProwessCode,Symbol,AsOnDate,ACP,pct,RF,RMRF,MF,SMB,HML,OLS210_intercept,OLS210_RMRF,OLS210_SMB,OLS210_HML,OLS210_r_squared,OLS210_adjusted_r_squared,OLS210_f_p_value,pct_cen3sum,RF_cen3sum,RMRF_cen3sum,SMB_cen3sum,HML_cen3sum,pct_cen5sum,RF_cen5sum,RMRF_cen5sum,SMB_cen5sum,HML_cen5sum,pct_cen7sum,RF_cen7sum,RMRF_cen7sum,SMB_cen7sum,HML_cen7sum,pct_cen11sum,RF_cen11sum,RMRF_cen11sum,SMB_cen11sum,HML_cen11sum,210CAR3,210CAR5,210CAR7,210CAR11
0,20 Microns Ltd.,11.0,20MICRONS,2011-04-29,20.48,-0.016330,0.019851,-0.595940,-0.576089,-0.572908,-0.607305,-0.022830,0.009052,0.012831,0.002836,0.131758,0.119114,2.058205e-06,0.135960,0.099051,-2.055273,-1.390252,0.761902,0.068979,0.138538,-4.268887,-1.176797,-0.293157,0.048651,0.178026,-4.607642,-1.723837,0.153421,0.066696,0.316887,-5.076389,-1.348640,-0.171170,0.139681,0.053503,0.002506,-0.117960
1,20 Microns Ltd.,11.0,20MICRONS,2014-08-06,31.15,-0.011111,0.022705,-0.690542,-0.667837,1.023324,-0.182707,-0.036365,0.006364,0.006151,0.003198,0.065439,0.051829,2.939000e-03,-0.012716,0.068115,-0.250139,1.214791,0.739524,-0.004716,0.158951,-0.558310,0.776436,0.169694,-0.002971,0.249787,-1.450041,2.056452,-0.913487,-0.040029,0.363107,-1.042710,0.259265,-3.508804,0.020019,-0.056337,-0.144163,-0.277777
2,20 Microns Ltd.,11.0,20MICRONS,2015-02-05,38.50,-0.020356,0.021755,-0.625772,-0.604017,0.012769,-1.213916,-0.035407,0.007727,0.003104,-0.001389,0.044002,0.030080,2.564648e-02,0.001722,0.065159,-1.736416,-0.804964,0.077852,-0.022663,0.152086,-2.821734,-1.126323,-2.075115,-0.020077,0.238803,-2.753260,0.046695,-2.249996,-0.086138,0.325611,-0.871457,0.052846,1.148929,0.058810,-0.046110,-0.134653,-0.297360
3,20 Microns Ltd.,11.0,20MICRONS,2017-05-04,43.70,0.016279,0.016554,0.046274,0.062828,0.831535,-0.581499,-0.024819,0.014223,0.010697,0.003037,0.206107,0.194546,2.516556e-10,-0.023826,0.049554,-0.994942,1.668469,-2.628765,-0.031524,0.165027,-0.704433,2.261335,-2.673886,-0.043378,0.198027,-0.312208,1.999131,-0.730319,-0.078551,0.263922,0.266193,0.943764,-0.758803,0.005364,-0.128143,-0.181674,-0.279592
4,20 Microns Ltd.,11.0,20MICRONS,2019-05-28,41.00,-0.016787,0.016873,0.367905,0.384778,-0.177790,0.292518,-0.024276,0.015716,0.012604,0.001488,0.334962,0.325277,3.803786e-18,0.060421,0.084375,0.593600,1.677151,-0.006311,0.127139,0.117695,2.441371,2.682737,0.292993,0.116773,0.151016,2.141790,1.865647,-0.441072,0.099791,0.250771,1.525870,1.719963,-0.901331,0.018415,0.009655,-0.017933,-0.122470
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33898,Zylog Systems Ltd.,275793.0,ZYLOG,2015-08-14,5.40,0.018868,0.019213,1.546309,1.565522,-0.064032,1.416502,-0.034644,0.006125,0.000900,0.012578,0.116788,0.103926,1.131949e-05,-0.113635,0.096076,0.535057,-0.082231,-0.294096,-0.246220,0.134926,-0.534668,-0.050669,-1.880907,-0.291712,0.173777,-0.844522,0.077811,-2.519018,-0.274151,0.291613,-3.755096,0.048701,-3.466074,-0.105282,-0.250236,-0.324769,-0.395281
33899,Zylog Systems Ltd.,275793.0,ZYLOG,2016-06-30,3.90,0.040000,0.017832,1.065921,1.083753,-0.809196,1.214423,-0.028389,0.004898,0.001269,0.005712,0.051949,0.038142,1.164031e-02,0.080218,0.053601,2.887396,-0.810372,2.264789,0.066209,0.125044,3.865878,-0.074499,3.413269,0.082876,0.196699,3.920553,2.024007,5.684697,0.017235,0.285435,1.822980,3.816329,1.394239,0.085732,-0.012006,-0.082900,-0.204769
33900,Zylog Systems Ltd.,275793.0,ZYLOG,2016-08-12,3.15,-0.045455,0.017406,0.290292,0.307698,-0.979555,0.368906,-0.029012,0.003331,-0.001272,0.004015,0.031630,0.017528,8.440892e-02,-0.061816,0.104454,0.121902,-1.936704,0.287865,-0.076967,0.139266,-1.118332,-2.021379,0.410728,-0.061583,0.174078,-0.982620,-0.933242,-0.384302,-0.059889,0.313345,0.230677,-0.096989,0.458040,-0.083260,-0.129692,-0.144996,-0.288929
33901,Zylog Systems Ltd.,275793.0,ZYLOG,2016-11-23,4.25,-0.022989,0.015807,1.118596,1.134402,0.640072,1.506896,-0.027645,0.001205,-0.

In [66]:
output210CAR_2.to_pickle(rf"{output_folder_path}\output210CAR_2.pkl")